# Smart Waste Sorter - recyclable waste image classification
**INFO813 Artificial Intelligence - Project (Trimester 3, 2026)**

This notebook runs the complete project pipeline, in the same order as the report:

| Notebook section | Report section |
|---|---|
| 1. Data acquisition and description | 1.1 Description and explanation |
| 2. Visualisation | 1.2 Visualisation |
| 3. Data transformation | 2.2 Data transformation |
| 4. Baseline models | 2.1 Justification of the modelling technique |
| 5. Training EfficientNet-B0 | 2.3 Implementation |
| 6. Evaluation | 2.4 Evaluation |
| 7. Export for the application | 3 Software implementation |

**How to run:** `Runtime > Change runtime type > T4 GPU`, then `Runtime > Run all`.
The first run takes about 20-30 minutes on a T4 GPU. When Google asks for permission to
access Google Drive, accept it (the dataset zip is read from Drive and the results are saved there).
At the end `results.zip` (figures, metrics, ONNX model, sample images) is saved to Drive and downloaded.


## 0. Environment

In [ ]:
# GPU check and extra libraries (PyTorch, scikit-learn, OpenCV and pandas are pre-installed in Colab)
!nvidia-smi -L || echo "No GPU found - use Runtime > Change runtime type > T4 GPU (the CPU also works, but slowly)"
%pip install -q timm imagehash onnx onnxruntime onnxscript kagglehub

### Project source code
The next cells write the `waste_classifier` Python package (identical to `src/waste_classifier`
in the GitHub repository). Each module is documented; the analysis cells further down only call
these functions.

In [ ]:
!mkdir -p waste_classifier

In [ ]:
%%writefile waste_classifier/__init__.py
"""Smart Waste Sorter - image classification of recyclable waste (INFO813 project).

Modules
-------
config          class names, image size, hyper-parameters
data            dataset discovery, file index, stratified group split, transforms
image_analysis  per-image statistics, perceptual hashing, duplicate detection
features        hand-crafted features (colour histogram, HOG, LBP) for baselines
models          EfficientNet-B0 (transfer learning) and a small CNN
train           two-phase training loop with early stopping
evaluate        metrics, bootstrap confidence intervals, calibration, robustness
explain         Grad-CAM heat-maps
export          ONNX export for the application
plots           every figure of the report
pipeline        the steps of the report, end to end
"""
__version__ = "1.0.0"


In [ ]:
%%writefile waste_classifier/config.py
"""Project-wide configuration.

Every value that controls the experiment lives here, so the data analysis,
the training code, the Colab notebook and the application all agree on the
class names, the image size, the normalisation constants and the random seed.
"""
from __future__ import annotations

from dataclasses import asdict, dataclass

# Fixed random seed -> reproducible splits, sampling and training.
SEED = 42

# The six material classes in alphabetical order.
# The integer label of a class is its position in this list (label encoding).
CLASS_NAMES = ["cardboard", "glass", "metal", "paper", "plastic", "trash"]
CLASS_TO_IDX = {name: idx for idx, name in enumerate(CLASS_NAMES)}

# File extensions that are treated as images.
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff")

# ImageNet channel statistics. The backbone network was pre-trained on ImageNet
# images normalised with these values, so our images must be normalised the same way.
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

# Input resolution of the network (EfficientNet-B0 native resolution).
IMG_SIZE = 224

# One fixed colour per class, used in every figure (colour follows the class).
# Categorical palette validated for colour-vision deficiency (adjacent pairs).
CLASS_COLORS = {
    "cardboard": "#2a78d6",
    "glass": "#eb6834",
    "metal": "#1baf7a",
    "paper": "#eda100",
    "plastic": "#e87ba4",
    "trash": "#008300",
}


@dataclass
class SplitConfig:
    """How the data are divided into training, validation and test sets."""

    test_size: float = 0.15      # share of images held out for the final test
    val_size: float = 0.15       # share of images used for model selection
    dup_threshold: int = 4       # pHash Hamming distance that defines a near-duplicate
    seed: int = SEED


@dataclass
class TrainConfig:
    """Hyper-parameters of the final model (EfficientNet-B0, transfer learning)."""

    model_name: str = "efficientnet_b0"
    img_size: int = IMG_SIZE
    batch_size: int = 64
    num_workers: int = 2
    # Phase 1 trains only the new classification head (backbone frozen).
    head_epochs: int = 3
    head_lr: float = 1e-3
    # Phase 2 fine-tunes the whole network with a smaller learning rate.
    finetune_epochs: int = 15
    finetune_lr: float = 3e-4
    weight_decay: float = 1e-2
    label_smoothing: float = 0.1
    drop_rate: float = 0.3        # dropout before the classifier
    drop_path_rate: float = 0.1   # stochastic depth inside the backbone
    patience: int = 4             # early stopping patience (epochs without improvement)
    seed: int = SEED

    def to_dict(self) -> dict:
        return asdict(self)


@dataclass
class BaselineConfig:
    """Settings of the comparison (baseline) models."""

    handcrafted_size: int = 128   # images are resized to 128x128 for HOG/colour/LBP features
    pca_components: int = 150
    cnn_img_size: int = 128       # small CNN trained from scratch
    cnn_epochs: int = 15
    cnn_lr: float = 1e-3
    cnn_batch_size: int = 64
    seed: int = SEED


In [ ]:
%%writefile waste_classifier/utils.py
"""Small helpers shared by the whole project (seeding, device, JSON, timing)."""
from __future__ import annotations

import json
import os
import platform
import random
import time
from contextlib import contextmanager
from pathlib import Path

import numpy as np


def set_seed(seed: int = 42) -> None:
    """Make Python, NumPy and PyTorch random generators reproducible."""
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch

        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except ImportError:  # torch is not needed for the data-analysis part
        pass


def get_device():
    """Return the fastest available PyTorch device: CUDA GPU, Apple MPS or CPU."""
    import torch

    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def _to_builtin(obj):
    """Convert NumPy / pandas scalars and arrays into JSON-serialisable Python types."""
    if isinstance(obj, dict):
        return {str(k): _to_builtin(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_to_builtin(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return _to_builtin(obj.tolist())
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return None if np.isnan(obj) else float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, float) and obj != obj:  # NaN -> null
        return None
    return obj


def save_json(data, path) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(_to_builtin(data), indent=2), encoding="utf-8")
    return path


def load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


@contextmanager
def timer(label: str, log=print):
    """Context manager that logs how long a block of code took."""
    start = time.perf_counter()
    yield
    log(f"[time] {label}: {time.perf_counter() - start:.1f} s")


def environment_info() -> dict:
    """Versions of the main libraries (reported in the notebook and the report)."""
    info = {"python": platform.python_version(), "platform": platform.platform()}
    for name in ["numpy", "pandas", "sklearn", "skimage", "cv2", "PIL", "matplotlib",
                 "torch", "torchvision", "timm", "onnx", "onnxruntime", "imagehash"]:
        try:
            module = __import__(name)
            info[name] = getattr(module, "__version__", "unknown")
        except Exception:  # library not installed in this environment
            info[name] = "not installed"
    try:
        import torch

        info["cuda_available"] = torch.cuda.is_available()
        if torch.cuda.is_available():
            info["gpu"] = torch.cuda.get_device_name(0)
    except Exception:
        pass
    return info


In [ ]:
%%writefile waste_classifier/data.py
"""Loading the image files, building the file index and creating the data splits."""
from __future__ import annotations

import shutil
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageOps

from .config import CLASS_NAMES, CLASS_TO_IDX, IMAGE_EXTENSIONS, IMAGENET_MEAN, IMAGENET_STD


# --------------------------------------------------------------------------------------
# Locating and indexing the dataset
# --------------------------------------------------------------------------------------
def extract_zip(zip_path, dest_dir) -> Path:
    """Extract the Kaggle archive (skipped if it was already extracted)."""
    zip_path, dest_dir = Path(zip_path), Path(dest_dir)
    marker = dest_dir / ".extracted"
    if marker.exists():
        return dest_dir
    dest_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(dest_dir)
    marker.write_text(zip_path.name)
    return dest_dir


def find_dataset_root(search_dir) -> Path:
    """Return the folder that contains one sub-folder per class (e.g. .../images)."""
    search_dir = Path(search_dir)
    wanted = set(CLASS_NAMES)
    candidates = [search_dir] + sorted(p for p in search_dir.rglob("*") if p.is_dir())
    for folder in candidates:
        subfolders = {p.name.lower() for p in folder.iterdir() if p.is_dir()}
        if len(subfolders & wanted) >= 4:  # tolerate a missing class, report it later
            return folder
    raise FileNotFoundError(
        f"No folder with class sub-folders {CLASS_NAMES} was found under {search_dir}"
    )


def build_index(root) -> pd.DataFrame:
    """One row per file: path, class name, integer label, extension and size on disk."""
    root = Path(root)
    rows = []
    for class_dir in sorted(p for p in root.iterdir() if p.is_dir()):
        class_name = class_dir.name.lower()
        if class_name not in CLASS_TO_IDX:
            print(f"[warning] folder '{class_dir.name}' is not one of the expected classes - skipped")
            continue
        for file in sorted(class_dir.rglob("*")):
            if not file.is_file() or file.name.startswith("."):
                continue
            rows.append(
                {
                    "path": str(file),
                    "rel_path": file.relative_to(root).as_posix(),
                    "filename": file.name,
                    "class_name": class_name,
                    "label": CLASS_TO_IDX[class_name],
                    "ext": file.suffix.lower(),
                    "file_size_kb": file.stat().st_size / 1024,
                }
            )
    df = pd.DataFrame(rows)
    df["is_image_file"] = df["ext"].isin(IMAGE_EXTENSIONS)
    return df


def find_metadata_csv(root) -> Path | None:
    """The Kaggle archive ships a metadata.csv next to (or above) the image folders."""
    root = Path(root)
    for folder in [root, root.parent, root.parent.parent]:
        hits = sorted(folder.glob("*.csv"))
        if hits:
            return hits[0]
    return None


def compare_with_metadata(index: pd.DataFrame, metadata_path) -> dict:
    """Cross-check the folder structure against metadata.csv (labels and file names)."""
    meta = pd.read_csv(metadata_path)
    report = {"metadata_file": Path(metadata_path).name, "metadata_rows": len(meta),
              "metadata_columns": list(meta.columns)}
    # Guess which column holds the file name and which holds the label.
    file_col = next((c for c in meta.columns if meta[c].astype(str).str.contains(r"\.(?:jpe?g|png)$", case=False).mean() > 0.5), None)
    label_col = next((c for c in meta.columns if set(meta[c].astype(str).str.lower().unique()) <= set(CLASS_NAMES) | {"nan"}), None)
    report["file_column"], report["label_column"] = file_col, label_col
    if label_col is not None:
        report["metadata_counts"] = meta[label_col].astype(str).str.lower().value_counts().to_dict()
    if file_col is not None:
        meta_names = set(meta[file_col].astype(str).map(lambda s: Path(s).name))
        disk_names = set(index["filename"])
        report["files_in_metadata_not_on_disk"] = len(meta_names - disk_names)
        report["files_on_disk_not_in_metadata"] = len(disk_names - meta_names)
        if label_col is not None:
            merged = index.merge(
                meta.assign(_name=meta[file_col].astype(str).map(lambda s: Path(s).name),
                            _label=meta[label_col].astype(str).str.lower())[["_name", "_label"]],
                left_on="filename", right_on="_name", how="inner")
            report["label_agreement"] = float((merged["_label"] == merged["class_name"]).mean()) if len(merged) else None
    return report


# --------------------------------------------------------------------------------------
# Train / validation / test split
# --------------------------------------------------------------------------------------
def make_splits(labels, groups, test_size=0.15, val_size=0.15, seed=42) -> np.ndarray:
    """Stratified *group* split into train / val / test.

    * Stratified: every split keeps the class proportions of the full dataset.
    * Grouped: near-duplicate images share a group id and always land in the same
      split, so the test set never contains a copy of a training image (no leakage).
    """
    from sklearn.model_selection import StratifiedGroupKFold

    labels, groups = np.asarray(labels), np.asarray(groups)
    n = len(labels)
    split = np.array(["train"] * n, dtype=object)

    n_test_folds = max(2, int(round(1 / test_size)))
    outer = StratifiedGroupKFold(n_splits=n_test_folds, shuffle=True, random_state=seed)
    trainval_idx, test_idx = next(outer.split(np.zeros(n), labels, groups))
    split[test_idx] = "test"

    relative_val = val_size / (1 - len(test_idx) / n)
    n_val_folds = max(2, int(round(1 / relative_val)))
    inner = StratifiedGroupKFold(n_splits=n_val_folds, shuffle=True, random_state=seed)
    _, val_rel = next(inner.split(np.zeros(len(trainval_idx)), labels[trainval_idx], groups[trainval_idx]))
    split[trainval_idx[val_rel]] = "val"
    return split


# --------------------------------------------------------------------------------------
# PyTorch dataset and image transformations
# --------------------------------------------------------------------------------------
def load_rgb(path) -> Image.Image:
    """Open an image, apply its EXIF orientation and convert it to 3-channel RGB."""
    with Image.open(path) as img:
        img = ImageOps.exif_transpose(img)
        return img.convert("RGB")


class RandomGaussianNoise:
    """Add camera-like Gaussian noise to a tensor in [0, 1] with probability `p`.

    The noise strength is drawn uniformly from [0, max_sigma] for every image, so the
    network learns to ignore the sensor noise of photos taken in poor light.
    """

    def __init__(self, p: float = 0.3, max_sigma: float = 0.08):
        self.p, self.max_sigma = p, max_sigma

    def __call__(self, x):
        import torch

        if torch.rand(1).item() < self.p:
            sigma = torch.rand(1).item() * self.max_sigma
            x = (x + torch.randn_like(x) * sigma).clamp(0.0, 1.0)
        return x


def _augmentations(img_size: int):
    """Random transformations applied to training images (PIL stage)."""
    from torchvision import transforms as T

    return [
        T.RandomResizedCrop(img_size, scale=(0.6, 1.0), ratio=(0.8, 1.25), interpolation=T.InterpolationMode.BICUBIC),
        T.RandomHorizontalFlip(p=0.5),
        T.RandomVerticalFlip(p=0.2),
        T.RandomRotation(degrees=20, interpolation=T.InterpolationMode.BILINEAR),
        T.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.25),  # hue unchanged: colour is informative
        T.RandomApply([T.GaussianBlur(kernel_size=5, sigma=(0.1, 1.5))], p=0.2),  # slightly out-of-focus photos
    ]


def build_transforms(img_size: int = 224, train: bool = False):
    """Image transformations applied before the images enter the network.

    Evaluation: resize the shorter side to `img_size`, centre-crop a square,
    convert to a tensor in [0, 1] and normalise with the ImageNet statistics.
    Training: the same plus random data augmentation (crop, flips, rotation,
    brightness/contrast/saturation jitter, blur and sensor noise) so the model sees a
    slightly different version of every image in every epoch, which reduces
    over-fitting. The hue is not changed because colour carries class information
    (brown cardboard, green glass).
    """
    from torchvision import transforms as T

    bicubic = T.InterpolationMode.BICUBIC
    normalise = T.Normalize(IMAGENET_MEAN, IMAGENET_STD)
    if train:
        return T.Compose([*_augmentations(img_size), T.ToTensor(), RandomGaussianNoise(p=0.3, max_sigma=0.08), normalise])
    return T.Compose([T.Resize(img_size, interpolation=bicubic), T.CenterCrop(img_size), T.ToTensor(), normalise])


def augmentation_preview(img_size: int = 224):
    """Training augmentation without normalisation (used only to plot examples)."""
    from torchvision import transforms as T

    return T.Compose([*_augmentations(img_size), T.ToTensor(), RandomGaussianNoise(p=0.3, max_sigma=0.08),
                      T.ToPILImage()])


def make_dataset(paths, labels, transform):
    """Create a torch Dataset lazily (keeps torch optional for the analysis part)."""
    import torch

    class ImageDataset(torch.utils.data.Dataset):
        def __init__(self, paths, labels, transform):
            self.paths = list(paths)
            self.labels = np.asarray(labels, dtype=np.int64)
            self.transform = transform

        def __len__(self):
            return len(self.paths)

        def __getitem__(self, i):
            return self.transform(load_rgb(self.paths[i])), int(self.labels[i])

    return ImageDataset(paths, labels, transform)


def make_loader(dataset, batch_size, shuffle, num_workers=2, device_type="cpu"):
    import torch

    return torch.utils.data.DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=(device_type == "cuda"),
        persistent_workers=num_workers > 0,
        drop_last=False,
    )


def copy_tree(src, dst):
    shutil.copytree(src, dst, dirs_exist_ok=True)


In [ ]:
%%writefile waste_classifier/image_analysis.py
"""Per-image properties used for the data analysis (Part 1 of the report).

For every image we record
  * file properties  : format, colour mode, width, height, frames, integrity, MD5
  * perceptual hash  : 64-bit pHash used to find (near-)duplicate images
  * pixel statistics : mean R/G/B, brightness, contrast, saturation, colourfulness,
                       sharpness, entropy, edge density, background whiteness,
                       hue distribution
These "variables" describe each image numerically, so they can be summarised,
visualised and compared between classes.
"""
from __future__ import annotations

import hashlib
from concurrent.futures import ProcessPoolExecutor

import numpy as np
import pandas as pd
from PIL import Image, ImageOps

HUE_BINS = 18          # 18 bins of 20 degrees each
CHANNEL_BINS = 32      # intensity histogram bins per colour channel
THUMB_SIZE = 64        # size of the thumbnails averaged into the "mean image" of a class
NUMERIC_FEATURES = [
    "file_size_kb", "mean_r", "mean_g", "mean_b", "brightness", "contrast", "saturation",
    "colourfulness", "sharpness", "entropy", "edge_density", "white_background",
]


def _phash_hex(img: Image.Image) -> str:
    """64-bit perceptual hash as a 16-character hex string (kept as text so that
    pandas never converts it to a float and loses precision)."""
    import imagehash

    return str(imagehash.phash(img))


def hashes_to_uint64(hex_hashes) -> np.ndarray:
    return np.array([int(h, 16) for h in hex_hashes], dtype=np.uint64)


def analyse_image(path: str) -> tuple[dict, np.ndarray | None]:
    """Return (properties, 64x64 thumbnail) for one image file."""
    import cv2

    row = {"path": path}
    try:
        raw = open(path, "rb").read()
        row["md5"] = hashlib.md5(raw).hexdigest()
        with Image.open(path) as probe:          # integrity check without decoding
            row.update(format=probe.format, mode=probe.mode,
                       width=probe.size[0], height=probe.size[1],
                       n_frames=getattr(probe, "n_frames", 1))
            probe.verify()
        with Image.open(path) as img:             # full decode
            img = ImageOps.exif_transpose(img).convert("RGB")
            arr = np.asarray(img, dtype=np.uint8)
            row["phash"] = _phash_hex(img)
            thumb = np.asarray(img.resize((THUMB_SIZE, THUMB_SIZE), Image.BILINEAR), dtype=np.uint8)
        row["readable"] = True
        row["error"] = ""
    except Exception as exc:  # corrupted / unreadable file
        row.update(readable=False, error=f"{type(exc).__name__}: {exc}")
        return row, None

    rgb = arr.astype(np.float32)
    r, g, b = rgb[..., 0], rgb[..., 1], rgb[..., 2]
    lum = 0.299 * r + 0.587 * g + 0.114 * b                      # perceived brightness
    hsv = cv2.cvtColor(arr, cv2.COLOR_RGB2HSV)                   # H: 0-179, S/V: 0-255
    gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)

    row["mean_r"], row["mean_g"], row["mean_b"] = float(r.mean()), float(g.mean()), float(b.mean())
    row["brightness"] = float(lum.mean() / 255)
    row["contrast"] = float(lum.std() / 255)
    row["saturation"] = float(hsv[..., 1].mean() / 255)
    # Colourfulness metric of Hasler & Suesstrunk (2003).
    rg, yb = r - g, 0.5 * (r + g) - b
    row["colourfulness"] = float(np.hypot(rg.std(), yb.std()) + 0.3 * np.hypot(rg.mean(), yb.mean()))
    # Sharpness: variance of the Laplacian (low values = blurry image).
    row["sharpness"] = float(cv2.Laplacian(gray, cv2.CV_64F).var())
    # Shannon entropy of the grey-level histogram (amount of detail / texture).
    hist = np.bincount(gray.ravel(), minlength=256) / gray.size
    nz = hist[hist > 0]
    row["entropy"] = float(-(nz * np.log2(nz)).sum())
    # Edge density: share of pixels on a Canny edge.
    row["edge_density"] = float((cv2.Canny(gray, 100, 200) > 0).mean())
    # Background whiteness: share of near-white, low-saturation pixels in a 10-px border.
    border = np.zeros(gray.shape, dtype=bool)
    border[:10, :] = border[-10:, :] = border[:, :10] = border[:, -10:] = True
    white = (lum > 200) & (hsv[..., 1] < 40)
    row["white_background"] = float(white[border].mean())
    # Hue histogram of clearly coloured pixels (share of all pixels in each 20-degree bin).
    coloured = (hsv[..., 1] >= 40) & (hsv[..., 2] >= 40)
    hue_counts = np.bincount((hsv[..., 0][coloured] // 10).astype(int), minlength=HUE_BINS)[:HUE_BINS]
    for i, c in enumerate(hue_counts):
        row[f"hue_{i:02d}"] = float(c / gray.size)
    # Intensity histogram of each colour channel (32 bins of 8 grey levels).
    for ch, name in enumerate("rgb"):
        counts = np.bincount((arr[..., ch] >> 3).ravel(), minlength=CHANNEL_BINS) / gray.size
        for i, c in enumerate(counts):
            row[f"{name}_hist_{i:02d}"] = float(c)
    return row, thumb


def _analyse_chunk(args):
    """Worker: analyse a chunk of images and return rows + per-class thumbnail sums."""
    paths, labels, n_classes = args
    rows, sums = [], np.zeros((n_classes, THUMB_SIZE, THUMB_SIZE, 3), dtype=np.float64)
    counts = np.zeros(n_classes, dtype=np.int64)
    for path, label in zip(paths, labels):
        row, thumb = analyse_image(path)
        rows.append(row)
        if thumb is not None:
            sums[label] += thumb
            counts[label] += 1
    return rows, sums, counts


def analyse_dataset(index: pd.DataFrame, n_classes: int, workers: int = 2, chunk: int = 250, log=print):
    """Analyse every image (in parallel) and compute the mean image of each class."""
    paths, labels = index["path"].tolist(), index["label"].to_numpy()
    jobs = [(paths[i:i + chunk], labels[i:i + chunk], n_classes) for i in range(0, len(paths), chunk)]
    all_rows = []
    sums = np.zeros((n_classes, THUMB_SIZE, THUMB_SIZE, 3), dtype=np.float64)
    counts = np.zeros(n_classes, dtype=np.int64)
    if workers > 1:
        with ProcessPoolExecutor(max_workers=workers) as pool:
            results = pool.map(_analyse_chunk, jobs)
            for k, (rows, s, c) in enumerate(results, 1):
                all_rows += rows; sums += s; counts += c
                if k % 10 == 0 or k == len(jobs):
                    log(f"  analysed {min(k * chunk, len(paths))}/{len(paths)} images")
    else:
        for k, job in enumerate(jobs, 1):
            rows, s, c = _analyse_chunk(job)
            all_rows += rows; sums += s; counts += c
    stats = pd.DataFrame(all_rows)
    stats = index.merge(stats, on="path", how="left")
    mean_images = (sums / np.maximum(counts, 1)[:, None, None, None]).astype(np.uint8)
    return stats, mean_images


# --------------------------------------------------------------------------------------
# Duplicate detection
# --------------------------------------------------------------------------------------
_POP8 = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)


def _popcount64(x: np.ndarray) -> np.ndarray:
    if hasattr(np, "bitwise_count"):          # NumPy >= 2.0
        return np.bitwise_count(x).astype(np.uint8)
    return _POP8[x.view(np.uint8)].reshape(*x.shape, 8).sum(-1).astype(np.uint8)


def near_duplicates(hashes, threshold: int = 4, block: int = 512):
    """Compare every pair of 64-bit pHashes.

    Returns
      pairs   : DataFrame (i, j, distance) with distance <= threshold (i < j)
      nn_dist : for each image, the Hamming distance to its most similar other image
    """
    h = np.asarray(hashes, dtype=np.uint64)
    n = len(h)
    nn_dist = np.full(n, 64, dtype=np.uint8)
    found = []
    for start in range(0, n, block):
        stop = min(start + block, n)
        d = _popcount64(np.bitwise_xor(h[start:stop, None], h[None, :]))
        d[np.arange(stop - start), np.arange(start, stop)] = 255      # ignore self-match
        nn_dist[start:stop] = d.min(axis=1)
        ii, jj = np.nonzero(d <= threshold)
        ii = ii + start
        keep = jj > ii
        found.append(np.stack([ii[keep], jj[keep], d[ii[keep] - start, jj[keep]]], axis=1))
    pairs = np.concatenate(found) if found else np.empty((0, 3), dtype=int)
    return pd.DataFrame(pairs, columns=["i", "j", "distance"]).astype(int), nn_dist


def duplicate_groups(n: int, pairs: pd.DataFrame) -> np.ndarray:
    """Union-find: images connected by near-duplicate pairs get the same group id."""
    parent = np.arange(n)

    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]
            a = parent[a]
        return a

    for a, b in zip(pairs["i"].to_numpy(), pairs["j"].to_numpy()):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[max(ra, rb)] = min(ra, rb)
    return np.array([find(a) for a in range(n)])


In [ ]:
%%writefile waste_classifier/features.py
"""Hand-crafted image features for the classical machine-learning baselines.

Classical algorithms (k-NN, logistic regression, random forest, SVM) cannot use raw
pixels effectively, so each image is converted into a fixed-length feature vector:
  * colour  : joint HSV colour histogram (8 hue x 4 saturation x 4 value = 128 bins)
  * shape   : Histogram of Oriented Gradients (HOG) on a 128x128 grey image (1,764 values)
  * texture : Local Binary Pattern (LBP) histogram (10 values)
"""
from __future__ import annotations

from concurrent.futures import ProcessPoolExecutor

import numpy as np
from PIL import Image


def handcrafted_features(path: str, size: int = 128) -> np.ndarray:
    import cv2
    from skimage.feature import hog, local_binary_pattern

    with Image.open(path) as img:
        arr = np.asarray(img.convert("RGB").resize((size, size), Image.BILINEAR), dtype=np.uint8)
    hsv = cv2.cvtColor(arr, cv2.COLOR_RGB2HSV)
    colour = cv2.calcHist([hsv], [0, 1, 2], None, [8, 4, 4], [0, 180, 0, 256, 0, 256]).ravel()
    colour = colour / max(colour.sum(), 1)
    gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
    shape = hog(gray, orientations=9, pixels_per_cell=(16, 16), cells_per_block=(2, 2),
                block_norm="L2-Hys", feature_vector=True)
    lbp = local_binary_pattern(gray, P=8, R=1, method="uniform")
    texture = np.histogram(lbp, bins=10, range=(0, 10))[0] / lbp.size
    return np.concatenate([colour, shape, texture]).astype(np.float32)


def _chunk(args):
    paths, size = args
    return np.stack([handcrafted_features(p, size) for p in paths])


def extract_features(paths, size: int = 128, workers: int = 2, chunk: int = 250) -> np.ndarray:
    paths = list(paths)
    jobs = [(paths[i:i + chunk], size) for i in range(0, len(paths), chunk)]
    if workers > 1:
        with ProcessPoolExecutor(max_workers=workers) as pool:
            parts = list(pool.map(_chunk, jobs))
    else:
        parts = [_chunk(job) for job in jobs]
    return np.concatenate(parts)


In [ ]:
%%writefile waste_classifier/models.py
"""Neural-network architectures: EfficientNet-B0 (transfer learning) and a small CNN."""
from __future__ import annotations

import os

import torch
from torch import nn

# Optional local copy of the ImageNet weights (used when the Hugging Face hub is not reachable).
WEIGHTS_ENV = "EFFICIENTNET_B0_WEIGHTS"


def _pretrained_kwargs(pretrained: bool) -> dict:
    kwargs = {"pretrained": pretrained}
    weights_file = os.environ.get(WEIGHTS_ENV)
    if pretrained and weights_file:
        kwargs["pretrained_cfg_overlay"] = {"file": weights_file}
    return kwargs


def create_efficientnet(num_classes: int, pretrained: bool = True,
                        drop_rate: float = 0.3, drop_path_rate: float = 0.1) -> nn.Module:
    """EfficientNet-B0 pre-trained on ImageNet with a new `num_classes` output layer.

    The convolutional backbone (feature extractor) keeps its ImageNet weights; only the
    final fully-connected layer is replaced, because ImageNet has 1,000 classes and our
    problem has six.
    """
    import timm

    model = timm.create_model("efficientnet_b0", num_classes=num_classes, drop_rate=drop_rate,
                              drop_path_rate=drop_path_rate, **_pretrained_kwargs(pretrained))
    # Start the new output layer at zero: every class starts with the same score (loss = ln 6)
    # instead of large random logits, so the head converges quickly in phase 1.
    head = model.get_classifier()
    nn.init.zeros_(head.weight)
    nn.init.zeros_(head.bias)
    return model


def create_feature_extractor(pretrained: bool = True) -> nn.Module:
    """EfficientNet-B0 without its classifier: returns a 1,280-d embedding per image."""
    import timm

    return timm.create_model("efficientnet_b0", num_classes=0, **_pretrained_kwargs(pretrained))


def set_backbone_trainable(model: nn.Module, trainable: bool) -> None:
    """Freeze (or unfreeze) every layer except the classification head."""
    head = model.get_classifier()
    head_params = {id(p) for p in head.parameters()}
    for p in model.parameters():
        if id(p) not in head_params:
            p.requires_grad = trainable
    for p in head.parameters():
        p.requires_grad = True


def count_parameters(model: nn.Module) -> tuple[int, int]:
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


class SimpleCNN(nn.Module):
    """A small VGG-style CNN trained from scratch (baseline without transfer learning).

    4 blocks of [conv 3x3 -> batch-norm -> ReLU] x 2 + max-pool, followed by global
    average pooling, dropout and a linear classifier (~1.2 M parameters).
    """

    def __init__(self, num_classes: int = 6):
        super().__init__()

        def block(c_in, c_out):
            return nn.Sequential(
                nn.Conv2d(c_in, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True),
                nn.Conv2d(c_out, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
            )

        self.features = nn.Sequential(block(3, 32), block(32, 64), block(64, 128), block(128, 256))
        self.classifier = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3),
                                        nn.Linear(256, num_classes))

    def forward(self, x):
        return self.classifier(self.features(x))


@torch.inference_mode()
def extract_embeddings(model: nn.Module, loader, device) -> tuple:
    """Run a frozen feature extractor over a loader and return (embeddings, labels)."""
    import numpy as np

    model.eval().to(device)
    feats, labels = [], []
    for x, y in loader:
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
            f = model(x.to(device, non_blocking=True))
        feats.append(f.float().cpu().numpy())
        labels.append(np.asarray(y))
    return np.concatenate(feats), np.concatenate(labels)


In [ ]:
%%writefile waste_classifier/train.py
"""Training loops: two-phase transfer learning with early stopping."""
from __future__ import annotations

import copy
import math
import time

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import f1_score
from torch import nn

from .models import count_parameters, set_backbone_trainable


def run_epoch(model, loader, criterion, device, optimizer=None, scheduler=None, scaler=None):
    """One pass over `loader`. Trains when an optimizer is given, otherwise evaluates.

    Returns (mean loss, accuracy, logits, labels); logits/labels only when evaluating.
    """
    training = optimizer is not None
    model.train(training)
    use_amp = device.type == "cuda"
    total_loss, correct, seen = 0.0, 0, 0
    all_logits, all_labels = [], []
    with torch.set_grad_enabled(training):
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                logits = model(x)
                loss = criterion(logits, y)
            if training:
                optimizer.zero_grad(set_to_none=True)
                if scaler is not None:
                    scaler.scale(loss).backward()
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    optimizer.step()
                if scheduler is not None:
                    scheduler.step()          # learning rate is updated every batch
            else:
                all_logits.append(logits.float().cpu())
                all_labels.append(y.cpu())
            total_loss += loss.item() * len(y)
            correct += (logits.argmax(1) == y).sum().item()
            seen += len(y)
    logits = torch.cat(all_logits).numpy() if all_logits else None
    labels = torch.cat(all_labels).numpy() if all_labels else None
    return total_loss / seen, correct / seen, logits, labels


def _make_scaler(device):
    if device.type != "cuda":
        return None
    try:
        return torch.amp.GradScaler("cuda")
    except (AttributeError, TypeError):  # older PyTorch
        return torch.cuda.amp.GradScaler()


def _cosine_with_warmup(optimizer, total_steps: int, warmup_steps: int):
    """Linear warm-up followed by cosine decay (per batch)."""
    def factor(step):
        if step < warmup_steps:
            return (step + 1) / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1 + math.cos(math.pi * min(1.0, progress)))

    return torch.optim.lr_scheduler.LambdaLR(optimizer, factor)


def _train_phase(model, phase, epochs, lr, train_loader, val_loader, criterion, device, cfg,
                 history, best, log, patience=None):
    """Train for up to `epochs` epochs, keeping the weights with the best validation macro-F1."""
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=cfg.weight_decay)
    steps = epochs * len(train_loader)
    scheduler = _cosine_with_warmup(optimizer, steps, warmup_steps=max(1, len(train_loader) // 2))
    scaler = _make_scaler(device)
    epochs_without_improvement = 0
    for epoch in range(1, epochs + 1):
        start = time.time()
        lr_now = optimizer.param_groups[0]["lr"]
        tr_loss, tr_acc, _, _ = run_epoch(model, train_loader, criterion, device, optimizer, scheduler, scaler)
        va_loss, va_acc, va_logits, va_labels = run_epoch(model, val_loader, criterion, device)
        va_f1 = f1_score(va_labels, va_logits.argmax(1), average="macro")
        row = {"epoch": len(history) + 1, "phase": phase, "lr": lr_now, "train_loss": tr_loss,
               "train_acc": tr_acc, "val_loss": va_loss, "val_acc": va_acc, "val_macro_f1": va_f1,
               "seconds": time.time() - start}
        history.append(row)
        improved = va_f1 > best["f1"] + 1e-4
        if improved:
            best.update(f1=va_f1, epoch=row["epoch"], state=copy.deepcopy(
                {k: v.detach().cpu() for k, v in model.state_dict().items()}))
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
        log(f"  [{phase}] epoch {row['epoch']:>2}: train loss {tr_loss:.3f} acc {tr_acc:.3f} | "
            f"val loss {va_loss:.3f} acc {va_acc:.3f} macro-F1 {va_f1:.3f}"
            f"{'  *best*' if improved else ''} ({row['seconds']:.0f}s)")
        if patience is not None and epochs_without_improvement >= patience:
            log(f"  early stopping: no improvement for {patience} epochs")
            break


def fit_transfer_learning(model, train_loader, val_loader, cfg, device, log=print):
    """Two-phase fine-tuning of a pre-trained network.

    Phase 1 ("head"): the ImageNet backbone is frozen and only the new classifier is
    trained, so the random output layer does not destroy the pre-trained features.
    Phase 2 ("fine-tune"): all layers are unfrozen and trained with a smaller learning
    rate, adapting the features to waste images. Early stopping keeps the weights of the
    epoch with the best validation macro-F1.
    """
    model.to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)
    history, best = [], {"f1": -1.0, "epoch": 0, "state": None}

    set_backbone_trainable(model, False)
    total, trainable = count_parameters(model)
    log(f"Phase 1 - frozen backbone: {trainable:,} of {total:,} parameters trainable")
    _train_phase(model, "head", cfg.head_epochs, cfg.head_lr, train_loader, val_loader,
                 criterion, device, cfg, history, best, log)

    set_backbone_trainable(model, True)
    total, trainable = count_parameters(model)
    log(f"Phase 2 - fine-tuning: {trainable:,} of {total:,} parameters trainable")
    _train_phase(model, "fine-tune", cfg.finetune_epochs, cfg.finetune_lr, train_loader, val_loader,
                 criterion, device, cfg, history, best, log, patience=cfg.patience)

    model.load_state_dict(best["state"])
    log(f"Best validation macro-F1 {best['f1']:.4f} at epoch {best['epoch']}")
    return model, pd.DataFrame(history), best["epoch"]


def fit_from_scratch(model, train_loader, val_loader, epochs, lr, device, weight_decay=1e-4,
                     patience=5, log=print):
    """Single-phase training used for the small CNN baseline."""
    class _Cfg:  # minimal config object for _train_phase
        pass

    cfg = _Cfg()
    cfg.weight_decay = weight_decay
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    history, best = [], {"f1": -1.0, "epoch": 0, "state": None}
    _train_phase(model, "scratch", epochs, lr, train_loader, val_loader, criterion, device, cfg,
                 history, best, log, patience=patience)
    model.load_state_dict(best["state"])
    return model, pd.DataFrame(history), best["epoch"]


@torch.inference_mode()
def predict_logits(model, loader, device) -> tuple[np.ndarray, np.ndarray]:
    criterion = nn.CrossEntropyLoss()
    _, _, logits, labels = run_epoch(model, loader, criterion, device)
    return logits, labels


In [ ]:
%%writefile waste_classifier/evaluate.py
"""Performance evaluation: metrics, confidence intervals, calibration and robustness."""
from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, cohen_kappa_score,
                             confusion_matrix, f1_score, log_loss, matthews_corrcoef,
                             precision_recall_fscore_support, roc_auc_score)


def softmax(logits: np.ndarray, temperature: float = 1.0) -> np.ndarray:
    z = logits / temperature
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


def expected_calibration_error(prob: np.ndarray, y: np.ndarray, n_bins: int = 15) -> float:
    """ECE: average gap between confidence and accuracy, weighted by bin size."""
    conf, pred = prob.max(1), prob.argmax(1)
    edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            ece += m.mean() * abs((pred[m] == y[m]).mean() - conf[m].mean())
    return float(ece)


def reliability_table(prob, y, n_bins: int = 10) -> pd.DataFrame:
    conf, pred = prob.max(1), prob.argmax(1)
    edges = np.linspace(0, 1, n_bins + 1)
    rows = []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        rows.append({"bin_low": lo, "bin_high": hi, "count": int(m.sum()),
                     "confidence": float(conf[m].mean()) if m.any() else np.nan,
                     "accuracy": float((pred[m] == y[m]).mean()) if m.any() else np.nan})
    return pd.DataFrame(rows)


def fit_temperature(logits: np.ndarray, y: np.ndarray) -> float:
    """Temperature scaling (Guo et al., 2017): one scalar T that minimises the
    validation negative log-likelihood. It changes confidence, never the predicted class."""
    from scipy.optimize import minimize_scalar

    def nll(log_t):
        p = softmax(logits, np.exp(log_t))
        return -np.log(np.clip(p[np.arange(len(y)), y], 1e-12, 1)).mean()

    res = minimize_scalar(nll, bounds=(-3, 3), method="bounded")
    return float(np.exp(res.x))


def classification_metrics(y: np.ndarray, prob: np.ndarray, class_names) -> dict:
    """Headline metrics for a multi-class classifier."""
    pred = prob.argmax(1)
    labels = np.arange(len(class_names))
    top2 = np.argsort(-prob, axis=1)[:, :2]
    p_macro, r_macro, f_macro, _ = precision_recall_fscore_support(y, pred, labels=labels,
                                                                   average="macro", zero_division=0)
    metrics = {
        "n": int(len(y)),
        "accuracy": accuracy_score(y, pred),
        "balanced_accuracy": balanced_accuracy_score(y, pred),
        "macro_precision": p_macro,
        "macro_recall": r_macro,
        "macro_f1": f_macro,
        "weighted_f1": f1_score(y, pred, average="weighted"),
        "cohen_kappa": cohen_kappa_score(y, pred),
        "mcc": matthews_corrcoef(y, pred),
        "top2_accuracy": float((top2 == y[:, None]).any(1).mean()),
        "log_loss": log_loss(y, np.clip(prob, 1e-12, 1), labels=labels),
        "ece": expected_calibration_error(prob, y),
    }
    try:
        metrics["roc_auc_ovr_macro"] = roc_auc_score(y, prob, multi_class="ovr", average="macro", labels=labels)
    except ValueError:  # a class is missing (only happens on tiny test runs)
        metrics["roc_auc_ovr_macro"] = float("nan")
    return {k: (float(v) if not isinstance(v, int) else v) for k, v in metrics.items()}


def per_class_report(y, prob, class_names) -> pd.DataFrame:
    pred = prob.argmax(1)
    labels = np.arange(len(class_names))
    p, r, f, s = precision_recall_fscore_support(y, pred, labels=labels, zero_division=0)
    df = pd.DataFrame({"class": class_names, "precision": p, "recall": r, "f1": f, "support": s})
    aucs = []
    for k in labels:
        try:
            aucs.append(roc_auc_score((y == k).astype(int), prob[:, k]))
        except ValueError:
            aucs.append(np.nan)
    df["roc_auc"] = aucs
    return df


def bootstrap_ci(y, pred, n_boot: int = 1000, seed: int = 42, alpha: float = 0.05) -> dict:
    """95% bootstrap confidence intervals for accuracy and macro-F1 on the test set."""
    rng = np.random.default_rng(seed)
    n = len(y)
    accs, f1s = [], []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        accs.append(accuracy_score(y[idx], pred[idx]))
        f1s.append(f1_score(y[idx], pred[idx], average="macro"))
    q = [100 * alpha / 2, 100 * (1 - alpha / 2)]
    return {"accuracy_ci": np.percentile(accs, q).tolist(), "macro_f1_ci": np.percentile(f1s, q).tolist()}


def confusion(y, prob, n_classes) -> np.ndarray:
    return confusion_matrix(y, prob.argmax(1), labels=np.arange(n_classes))


def most_confused_pairs(cm: np.ndarray, class_names, k: int = 5) -> pd.DataFrame:
    rows = []
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            if i != j and cm[i, j] > 0:
                rows.append({"true": class_names[i], "predicted": class_names[j], "count": int(cm[i, j]),
                             "share_of_true_class": cm[i, j] / cm[i].sum()})
    return pd.DataFrame(rows).sort_values("count", ascending=False).head(k).reset_index(drop=True)


def selective_prediction(prob, y, thresholds=None) -> pd.DataFrame:
    """Accuracy vs coverage when predictions below a confidence threshold are rejected."""
    thresholds = np.round(np.arange(0.0, 1.0, 0.05), 2) if thresholds is None else thresholds
    conf, correct = prob.max(1), prob.argmax(1) == y
    rows = []
    for t in thresholds:
        keep = conf >= t
        rows.append({"threshold": float(t), "coverage": float(keep.mean()),
                     "accuracy": float(correct[keep].mean()) if keep.any() else np.nan,
                     "rejected": int((~keep).sum())})
    return pd.DataFrame(rows)


# --------------------------------------------------------------------------------------
# Robustness to realistic image corruptions (test set)
# --------------------------------------------------------------------------------------
def corruption_transforms():
    """Simple corruptions that mimic phone photos taken in less ideal conditions."""
    from PIL import Image, ImageEnhance, ImageFilter

    def jpeg(img, quality=20):
        import io

        buf = io.BytesIO()
        img.save(buf, format="JPEG", quality=quality)
        buf.seek(0)
        return Image.open(buf).convert("RGB")

    def noise(img, sigma=20, seed=0):
        arr = np.asarray(img, dtype=np.float32)
        arr = arr + np.random.default_rng(seed).normal(0, sigma, arr.shape)
        return Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))

    return {
        "original": lambda im: im,
        "blur": lambda im: im.filter(ImageFilter.GaussianBlur(radius=2)),
        "darker (-40%)": lambda im: ImageEnhance.Brightness(im).enhance(0.6),
        "brighter (+40%)": lambda im: ImageEnhance.Brightness(im).enhance(1.4),
        "rotated 90 deg": lambda im: im.rotate(90, expand=True),
        "JPEG quality 20": jpeg,
        "Gaussian noise": noise,
        "greyscale": lambda im: im.convert("L").convert("RGB"),
    }


In [ ]:
%%writefile waste_classifier/explain.py
"""Grad-CAM (Selvaraju et al., 2017): which image regions drove a prediction."""
from __future__ import annotations

import numpy as np
import torch
import torch.nn.functional as F


class GradCAM:
    """Class-activation heat-map from the gradients of the last convolutional layer."""

    def __init__(self, model, target_layer=None):
        self.model = model.eval()
        # Grad-CAM needs the *activated* feature maps of the last convolutional stage. In timm's
        # EfficientNet, conv_head is followed by bn2 = BatchNorm + SiLU, so bn2's output is used.
        # (conv_head alone gives pre-activation values, whose negative parts invert the heat-map.)
        if target_layer is not None:
            layer = target_layer
        elif hasattr(model, "bn2"):
            layer = model.bn2
        else:
            layer = model.conv_head
        self.activations = None
        self.gradients = None
        self._handle = layer.register_forward_hook(self._save)

    def _save(self, module, inputs, output):
        self.activations = output
        if output.requires_grad:  # no gradient hooks during normal (inference) use
            output.register_hook(lambda grad: setattr(self, "gradients", grad))

    def remove(self):
        """Detach the hook so the model behaves normally afterwards."""
        self._handle.remove()

    def __call__(self, x: torch.Tensor, class_idx=None) -> tuple[np.ndarray, np.ndarray]:
        """x: (N,3,H,W) normalised batch. Returns (heat-maps N x H x W in [0,1], predicted classes)."""
        with torch.enable_grad():
            x = x.clone().requires_grad_(True)
            logits = self.model(x)
            pred = logits.argmax(1)
            target = pred if class_idx is None else torch.as_tensor(class_idx, device=x.device)
            self.model.zero_grad(set_to_none=True)
            logits.gather(1, target.view(-1, 1)).sum().backward()
            weights = self.gradients.mean(dim=(2, 3), keepdim=True)          # importance of each channel
            cam = F.relu((weights * self.activations).sum(dim=1, keepdim=True))
            cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[:, 0]
            cam = cam - cam.amin(dim=(1, 2), keepdim=True)
            cam = cam / cam.amax(dim=(1, 2), keepdim=True).clamp_min(1e-8)
        return cam.detach().cpu().numpy(), pred.detach().cpu().numpy()


def denormalise(x: torch.Tensor, mean, std) -> np.ndarray:
    """Undo ImageNet normalisation -> (N,H,W,3) float image in [0,1] for plotting."""
    mean = torch.tensor(mean).view(1, 3, 1, 1)
    std = torch.tensor(std).view(1, 3, 1, 1)
    img = (x.detach().cpu() * std + mean).clamp(0, 1)
    return img.permute(0, 2, 3, 1).numpy()


In [ ]:
%%writefile waste_classifier/export.py
"""Export the trained PyTorch model to ONNX so the application does not need PyTorch."""
from __future__ import annotations

import inspect
import time
from pathlib import Path

import numpy as np
import torch


def export_onnx(model, out_path, img_size: int = 224, opset: int = 17) -> Path:
    """Write the model as an ONNX graph with a dynamic batch dimension."""
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    model = model.eval().cpu()
    dummy = torch.randn(1, 3, img_size, img_size)
    kwargs = dict(input_names=["input"], output_names=["logits"], opset_version=opset,
                  dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}}, do_constant_folding=True)
    if "dynamo" in inspect.signature(torch.onnx.export).parameters:
        kwargs["dynamo"] = False  # classic exporter: simple, well-tested graphs for CNNs
    try:
        torch.onnx.export(model, (dummy,), str(out_path), **kwargs)
    except Exception as exc:  # newer PyTorch without the classic exporter -> dynamo exporter
        print(f"[export] classic exporter failed ({type(exc).__name__}); retrying with dynamo=True")
        kwargs.pop("do_constant_folding", None)
        kwargs["dynamo"] = True
        kwargs["opset_version"] = max(opset, 18)
        torch.onnx.export(model, (dummy,), str(out_path), **kwargs)
    _merge_external_data(out_path)
    return out_path


def _merge_external_data(path: Path) -> None:
    """Make sure all weights are stored inside the .onnx file (single-file model)."""
    import onnx

    model = onnx.load(str(path), load_external_data=True)
    onnx.save_model(model, str(path), save_as_external_data=False)
    for extra in path.parent.glob(path.name + ".data"):
        extra.unlink()


def compress_weights_fp16(src, dst, min_elements: int = 1024) -> Path:
    """Store large weight tensors as float16 (half the file size) + Cast nodes back to float32.

    ONNX Runtime folds the Cast nodes when the model is loaded, so inference still runs
    in float32; only the rounding of the stored weights changes (verified afterwards).
    """
    import onnx
    from onnx import TensorProto, helper, numpy_helper

    model = onnx.load(str(src))
    graph = model.graph
    graph_inputs = {i.name for i in graph.input}
    casts, new_inits = [], []
    for init in list(graph.initializer):
        if init.data_type != TensorProto.FLOAT or init.name in graph_inputs:
            continue
        if int(np.prod(init.dims)) < min_elements:
            continue
        arr = numpy_helper.to_array(init).astype(np.float16)
        half_name = init.name + "__fp16"
        new_inits.append(numpy_helper.from_array(arr, half_name))
        casts.append(helper.make_node("Cast", [half_name], [init.name], to=TensorProto.FLOAT,
                                      name=init.name + "__to_fp32"))
        graph.initializer.remove(init)
    graph.initializer.extend(new_inits)
    nodes = list(graph.node)
    del graph.node[:]
    graph.node.extend(casts + nodes)
    onnx.checker.check_model(model)
    onnx.save_model(model, str(dst))
    return Path(dst)


def onnx_logits(onnx_path, batch: np.ndarray, chunk: int = 32) -> np.ndarray:
    """Logits of an ONNX model for a batch of images, computed in small chunks to limit memory."""
    import onnxruntime as ort

    options = ort.SessionOptions()
    options.enable_cpu_mem_arena = False      # return memory after each run (Colab has ~12 GB RAM)
    session = ort.InferenceSession(str(onnx_path), sess_options=options, providers=["CPUExecutionProvider"])
    name = session.get_inputs()[0].name
    out = [session.run(None, {name: batch[i:i + chunk].astype(np.float32)})[0] for i in range(0, len(batch), chunk)]
    del session
    return np.concatenate(out)


def benchmark_latency(onnx_path, img_size: int = 224, runs: int = 30) -> dict:
    """Median CPU latency for one image (what the lecturer will experience)."""
    import onnxruntime as ort

    session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
    name = session.get_inputs()[0].name
    x = np.random.rand(1, 3, img_size, img_size).astype(np.float32)
    for _ in range(3):
        session.run(None, {name: x})
    times = []
    for _ in range(runs):
        start = time.perf_counter()
        session.run(None, {name: x})
        times.append((time.perf_counter() - start) * 1000)
    return {"median_ms": float(np.median(times)), "p90_ms": float(np.percentile(times, 90))}


In [ ]:
%%writefile waste_classifier/plots.py
"""All figures used in the report (consistent style: thin marks, recessive grid,
one fixed colour per class, colour-vision-deficiency-safe palette)."""
from __future__ import annotations

from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
from matplotlib.colors import LinearSegmentedColormap  # noqa: E402
from matplotlib.ticker import FuncFormatter, MaxNLocator, NullFormatter  # noqa: E402

from .config import CLASS_COLORS, CLASS_NAMES  # noqa: E402

# ---- design tokens --------------------------------------------------------------------
SURFACE = "#ffffff"
INK = "#0b0b0b"
INK_2 = "#52514e"
MUTED = "#898781"
GRID = "#e1e0d9"
AXIS = "#c3c2b7"
GREY_MARK = "#d6d5ce"
ACCENT = "#2a78d6"        # categorical slot 1 (single-series charts)
ACCENT_2 = "#eb6834"      # categorical slot 2
BLUE_RAMP = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]
SEQ = LinearSegmentedColormap.from_list("seq_blue", ["#f4f8fe"] + BLUE_RAMP)
DIV = LinearSegmentedColormap.from_list("div", ["#184f95", "#6da7ec", "#f0efec", "#ec7b7a", "#b52e2e"])
SPLIT_COLORS = {"train": "#184f95", "val": "#5598e7", "test": "#9ec5f4"}


def set_style() -> None:
    plt.rcParams.update({
        "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
        "axes.edgecolor": AXIS, "axes.linewidth": 0.8, "axes.labelcolor": INK_2,
        "axes.titlecolor": INK, "axes.titlesize": 11.5, "axes.titleweight": "bold",
        "axes.titlelocation": "left", "axes.titlepad": 10, "axes.labelsize": 9.5,
        "xtick.color": AXIS, "ytick.color": AXIS, "xtick.labelcolor": INK_2, "ytick.labelcolor": INK_2,
        "xtick.labelsize": 8.5, "ytick.labelsize": 8.5,
        "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.7, "grid.linestyle": "-",
        "axes.axisbelow": True, "axes.spines.top": False, "axes.spines.right": False,
        "font.family": "DejaVu Sans", "font.size": 9.5, "legend.frameon": False,
        "legend.fontsize": 8.5, "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
        "savefig.pad_inches": 0.15, "lines.linewidth": 2, "lines.solid_capstyle": "round",
    })


set_style()


def _save(fig, out) -> Path:
    out = Path(out)
    out.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out)
    plt.close(fig)
    return out


def _suptitle(fig, title, subtitle=None):
    fig.suptitle(title, x=0.01, ha="left", fontsize=13, fontweight="bold", color=INK, y=1.0)
    if subtitle:
        fig.text(0.01, 0.955, subtitle, ha="left", va="top", fontsize=9.5, color=INK_2)


def _no_grid_x(ax):
    ax.grid(axis="x", visible=False)


def _no_grid_y(ax):
    ax.grid(axis="y", visible=False)


# =======================================================================================
# Part 1 - data analysis
# =======================================================================================
def class_distribution(counts: pd.Series, out, title="Images per class"):
    """Column chart of the number of images in each class (categorical variable)."""
    counts = counts.reindex(CLASS_NAMES).fillna(0)
    total = counts.sum()
    fig, ax = plt.subplots(figsize=(7.5, 3.8))
    bars = ax.bar(counts.index, counts.values, width=0.55, color=ACCENT, zorder=2)
    for bar, value in zip(bars, counts.values):     # labels inside the bars, clear of the mean line
        ax.text(bar.get_x() + bar.get_width() / 2, value - counts.max() * 0.025, f"{int(value):,}\n{value / total:.1%}",
                ha="center", va="top", fontsize=8.5, color="white", fontweight="bold", zorder=3)
    mean = counts.mean()
    ax.axhline(mean, color=MUTED, lw=1, zorder=1)
    ax.set_xlim(-0.6, len(counts) - 0.4 + 0.75)            # room for the label right of the last bar
    ax.text(len(counts) - 0.32, mean, f"mean\n{mean:,.0f}", va="center", ha="left", fontsize=8, color=MUTED)
    ax.set_ylabel("Number of images")
    ax.set_ylim(0, counts.max() * 1.22)
    _no_grid_x(ax)
    ax.set_title(title)
    return _save(fig, out)


def duplicates_by_class(stats: pd.DataFrame, out):
    """Unique images vs redundant exact copies per class (stacked horizontal bars).

    A file is a 'redundant copy' when another file with the same MD5 checksum exists;
    one file of every identical group is counted as unique.
    """
    df = stats[["class_name", "md5"]].copy()
    per_class = df.groupby("class_name").agg(files=("md5", "size"), unique=("md5", "nunique")).reindex(CLASS_NAMES)
    per_class["copies"] = per_class["files"] - per_class["unique"]
    fig, ax = plt.subplots(figsize=(8.6, 3.6))
    y = np.arange(len(per_class))
    ax.barh(y, per_class["unique"], height=0.58, color=ACCENT, label="unique images", zorder=2)
    ax.barh(y, per_class["copies"], left=per_class["unique"], height=0.58, color=ACCENT_2,
            label="redundant exact copies", zorder=2)
    xmax = per_class["files"].max()
    for i, r in enumerate(per_class.itertuples()):
        ax.text(r.unique / 2, i, f"{r.unique:,}", ha="center", va="center", fontsize=8.5, color="white")
        share = r.copies / r.files if r.files else 0
        ax.text(r.files + xmax * 0.012, i, f"+{r.copies:,} copies ({share:.0%})" if r.copies else "no copies",
                ha="left", va="center", fontsize=8.5, color=INK_2)
    ax.set_yticks(y, per_class.index)
    ax.invert_yaxis()
    ax.set_xlim(0, xmax * 1.28)
    ax.set_xlabel("Number of files")
    ax.legend(loc="upper center", bbox_to_anchor=(0.45, -0.2), ncol=2)
    _no_grid_y(ax)
    ax.set_title("Unique images and byte-identical copies per class")
    return _save(fig, out)


def sample_grid(index: pd.DataFrame, out, per_class=6, seed=42):
    """Random examples of every class (qualitative inspection)."""
    from PIL import Image

    rng = np.random.default_rng(seed)
    fig, axes = plt.subplots(len(CLASS_NAMES), per_class, figsize=(per_class * 1.35, len(CLASS_NAMES) * 1.42))
    for r, cls in enumerate(CLASS_NAMES):
        paths = index.loc[index["class_name"] == cls, "path"].to_numpy()
        chosen = rng.choice(paths, size=min(per_class, len(paths)), replace=False) if len(paths) else []
        for c in range(per_class):
            ax = axes[r, c]
            ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
            for s in ax.spines.values():
                s.set_visible(False)
            if c < len(chosen):
                with Image.open(chosen[c]) as im:
                    ax.imshow(im.convert("RGB"))
            if c == 0:
                ax.set_ylabel(cls, rotation=0, ha="right", va="center", fontsize=10, color=INK, labelpad=8)
    fig.subplots_adjust(wspace=0.04, hspace=0.06)
    _suptitle(fig, "Random sample images from each class")
    return _save(fig, out)


def file_properties(stats: pd.DataFrame, out):
    """Resolution, file format, colour mode and file size (file-level variables)."""
    fig, axes = plt.subplots(1, 4, figsize=(13, 3.3), gridspec_kw={"width_ratios": [1.1, 0.8, 0.8, 1.5]})
    # (a) resolution
    ax = axes[0]
    dims = stats.groupby(["width", "height"]).size().reset_index(name="n")
    ax.scatter(dims["width"], dims["height"], s=40 + 260 * dims["n"] / dims["n"].max(), color=ACCENT,
               edgecolor=SURFACE, linewidth=2, zorder=3)
    for _, row in dims.nlargest(3, "n").iterrows():
        ax.annotate(f"{int(row.width)}x{int(row.height)}\n{int(row.n):,} images", (row.width, row.height),
                    xytext=(8, 8), textcoords="offset points", fontsize=8, color=INK_2)
    pad = max(20, (dims["width"].max() - dims["width"].min()) * 0.2)
    ax.set_xlim(dims["width"].min() - pad, dims["width"].max() + pad * 2.5)
    ax.set_ylim(dims["height"].min() - pad, dims["height"].max() + pad * 2.5)
    ax.set_xlabel("Width (px)"); ax.set_ylabel("Height (px)"); ax.set_title("(a) Resolution")
    # (b) format and (c) mode
    for ax, col, title in [(axes[1], "format", "(b) File format"), (axes[2], "mode", "(c) Colour mode")]:
        vc = stats[col].fillna("unreadable").value_counts()
        ax.bar(vc.index.astype(str), vc.values, width=0.5, color=ACCENT, zorder=2)
        for i, v in enumerate(vc.values):
            ax.text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=8, color=INK_2)
        ax.set_ylim(0, vc.max() * 1.18); ax.set_title(title); _no_grid_x(ax)
    # (d) file size
    ax = axes[3]
    ax.hist(stats["file_size_kb"], bins=50, color=ACCENT, edgecolor=SURFACE, linewidth=0.6, zorder=2)
    med = stats["file_size_kb"].median()
    ax.axvline(med, color=INK_2, lw=1)
    ax.text(med, ax.get_ylim()[1] * 0.95, f"  median {med:.1f} KB", fontsize=8, color=INK_2, va="top")
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax.set_xlabel("File size (KB)"); ax.set_ylabel("Images"); ax.set_title("(d) File size")
    fig.tight_layout()
    return _save(fig, out)


FEATURE_LABELS = {
    "file_size_kb": "File size (KB)", "brightness": "Brightness (0-1)", "contrast": "Contrast (0-1)",
    "saturation": "Saturation (0-1)", "colourfulness": "Colourfulness", "sharpness": "Sharpness (log10 var. Laplacian)",
    "entropy": "Entropy (bits)", "edge_density": "Edge density", "white_background": "White-background share of border",
    "mean_r": "Mean red", "mean_g": "Mean green", "mean_b": "Mean blue",
}


SHORT_LABELS = {
    "file_size_kb": "File size", "brightness": "Brightness", "contrast": "Contrast", "saturation": "Saturation",
    "colourfulness": "Colourfulness", "sharpness": "Sharpness", "entropy": "Entropy", "edge_density": "Edge density",
    "white_background": "White background", "mean_r": "Mean red", "mean_g": "Mean green", "mean_b": "Mean blue",
}


def feature_boxplots(stats: pd.DataFrame, features, out, title="Image statistics by class"):
    """Box plots of each numeric image variable, split by class."""
    n = len(features)
    cols = 4 if n % 4 == 0 else 3
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(3.4 * cols, 2.7 * rows), squeeze=False)
    for ax, feat in zip(axes.ravel(), features):
        data = []
        for cls in CLASS_NAMES:
            v = stats.loc[stats["class_name"] == cls, feat].dropna().to_numpy()
            data.append(np.log10(v + 1) if feat == "sharpness" else v)
        bp = ax.boxplot(data, vert=False, widths=0.55, patch_artist=True, showfliers=False,
                        medianprops={"color": INK, "linewidth": 1.4},
                        whiskerprops={"color": MUTED, "linewidth": 1}, capprops={"color": MUTED, "linewidth": 1})
        for patch, cls in zip(bp["boxes"], CLASS_NAMES):
            patch.set_facecolor(CLASS_COLORS[cls]); patch.set_alpha(0.55); patch.set_edgecolor(SURFACE)
        ax.set_yticks(range(1, len(CLASS_NAMES) + 1), CLASS_NAMES)
        ax.invert_yaxis()
        ax.set_title(FEATURE_LABELS.get(feat, feat), fontsize=10)
        ax.xaxis.set_major_locator(MaxNLocator(nbins=5))
        _no_grid_y(ax)
    for ax in axes.ravel()[n:]:
        ax.set_visible(False)
    _suptitle(fig, title, "Boxes show the middle 50% of images, the line is the median, whiskers 1.5 x IQR (outliers hidden)")
    fig.tight_layout(rect=(0, 0, 1, 0.94))
    return _save(fig, out)


def white_background_share(stats: pd.DataFrame, out, cut=0.6):
    """Share of images per class whose border is mostly plain white (studio-style photo)."""
    share = stats.assign(white=stats["white_background"] >= cut).groupby("class_name")["white"].mean().reindex(CLASS_NAMES)
    fig, ax = plt.subplots(figsize=(8, 3.4))
    ax.barh(share.index, share.values, height=0.55, color=ACCENT, zorder=2)
    for i, v in enumerate(share.values):
        ax.text(v + 0.01, i, f"{v:.1%}", va="center", fontsize=8.5, color=INK_2)
    ax.invert_yaxis()
    ax.set_xlim(0, max(0.2, share.max() * 1.25))
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:.0%}"))
    ax.set_xlabel(f"Images whose 10-px border is >= {cut:.0%} near-white pixels")
    _no_grid_y(ax)
    ax.set_title("Plain white (studio) backgrounds by class")
    return _save(fig, out)


def channel_histograms(stats: pd.DataFrame, out, bins=32):
    """Average red / green / blue intensity distribution of each class (small multiples)."""
    fig, axes = plt.subplots(2, 3, figsize=(12, 5.6), sharex=True, sharey=True)
    x = (np.arange(bins) + 0.5) * (256 / bins)
    channel_colors = {"r": "#d03b3b", "g": "#1f8f3a", "b": "#2a78d6"}
    names = {"r": "red", "g": "green", "b": "blue"}
    for ax, cls in zip(axes.ravel(), CLASS_NAMES):
        sub = stats[stats["class_name"] == cls]
        for ch in "rgb":
            cols = [f"{ch}_hist_{i:02d}" for i in range(bins)]
            ax.plot(x, sub[cols].mean().to_numpy() * 100, color=channel_colors[ch], lw=1.8, label=names[ch])
        ax.set_title(cls, fontsize=10.5)
        ax.set_xlim(0, 255)
    for ax in axes[-1]:
        ax.set_xlabel("Pixel intensity (0 = dark, 255 = bright)")
    for ax in axes[:, 0]:
        ax.set_ylabel("% of pixels")
    axes[0, 0].legend(loc="upper left")
    _suptitle(fig, "Colour-channel intensity distributions per class",
              "Average histogram of the red, green and blue channels over all images of the class")
    fig.tight_layout(rect=(0, 0, 1, 0.93))
    return _save(fig, out)


def hue_heatmap(stats: pd.DataFrame, out, n_bins=18):
    """Heat-map of the hue distribution (which colours dominate each class)."""
    import colorsys

    cols = [f"hue_{i:02d}" for i in range(n_bins)]
    mat = stats.groupby("class_name")[cols].mean().reindex(CLASS_NAMES).to_numpy() * 100
    fig = plt.figure(figsize=(11, 3.9))
    ax = fig.add_axes([0.1, 0.3, 0.78, 0.5])
    strip = fig.add_axes([0.1, 0.2, 0.78, 0.045])
    cax = fig.add_axes([0.9, 0.3, 0.015, 0.5])
    im = ax.imshow(mat, aspect="auto", cmap=SEQ, vmin=0)
    ax.set_yticks(range(len(CLASS_NAMES)), CLASS_NAMES)
    ax.set_xticks([]); ax.grid(False)
    for s in ax.spines.values():
        s.set_visible(False)
    hues = np.array([[colorsys.hsv_to_rgb((i * 20 + 10) / 360, 0.75, 0.9) for i in range(n_bins)]])
    strip.imshow(hues, aspect="auto")
    strip.set_yticks([])
    strip.set_xticks(np.arange(-0.5, n_bins, 3), [f"{d}°" for d in range(0, 361, 60)])
    strip.set_xlabel("Hue (colour wheel angle)")
    for s in strip.spines.values():
        s.set_visible(False)
    strip.grid(False)
    cb = fig.colorbar(im, cax=cax)
    cb.set_label("% of pixels", color=INK_2); cb.outline.set_visible(False)
    fig.text(0.01, 0.97, "Hue distribution of coloured pixels per class", fontsize=13, fontweight="bold", va="top")
    fig.text(0.01, 0.9, "Only pixels with saturation and value >= 40/255 are counted (greys and whites excluded)",
             fontsize=9.5, color=INK_2, va="top")
    return _save(fig, out)


def mean_images(mean_imgs: np.ndarray, out):
    """The pixel-wise average image of every class."""
    fig, axes = plt.subplots(1, len(CLASS_NAMES), figsize=(12, 2.6))
    for ax, cls, img in zip(axes, CLASS_NAMES, mean_imgs):
        ax.imshow(img); ax.set_title(cls, fontsize=10.5, loc="center")
        ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
        for s in ax.spines.values():
            s.set_visible(False)
    _suptitle(fig, "Average image of each class")
    fig.tight_layout(rect=(0, 0, 1, 0.9))
    return _save(fig, out)


def correlation_heatmap(stats: pd.DataFrame, features, out):
    """Spearman correlation between the numeric image variables."""
    corr = stats[features].corr(method="spearman")
    labels = [SHORT_LABELS.get(f, f) for f in features]
    fig, ax = plt.subplots(figsize=(8.2, 7))
    im = ax.imshow(corr.to_numpy(), cmap=DIV, vmin=-1, vmax=1)
    ax.set_xticks(range(len(features)), labels, rotation=45, ha="right")
    ax.set_yticks(range(len(features)), labels)
    ax.grid(False)
    for s in ax.spines.values():
        s.set_visible(False)
    vals = corr.to_numpy()
    for i in range(len(features)):
        for j in range(len(features)):
            if i != j and abs(vals[i, j]) >= 0.5:
                ax.text(j, i, f"{vals[i, j]:.2f}", ha="center", va="center", fontsize=7.5,
                        color="white" if abs(vals[i, j]) > 0.75 else INK)
    cb = fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
    cb.set_label("Spearman correlation", color=INK_2); cb.outline.set_visible(False)
    ax.set_title("Correlation between image variables (|r| >= 0.5 labelled)")
    return _save(fig, out)


def scatter_by_class(stats: pd.DataFrame, x, y, out, per_class=700, seed=42, logx=False, logy=False):
    """Relationship between two variables; one panel per class highlighted over all others."""
    rng = np.random.default_rng(seed)
    sample = pd.concat([d.sample(min(per_class, len(d)), random_state=int(rng.integers(1e9)))
                        for _, d in stats.groupby("class_name")])
    fig, axes = plt.subplots(2, 3, figsize=(12, 6.8), sharex=True, sharey=True)
    xs, ys = sample[x], sample[y]
    if logx:
        xs = np.log10(xs + 1)
    if logy:
        ys = np.log10(ys + 1)
    for ax, cls in zip(axes.ravel(), CLASS_NAMES):
        m = (sample["class_name"] == cls).to_numpy()
        ax.scatter(xs[~m], ys[~m], s=5, color=GREY_MARK, alpha=0.6, linewidth=0, rasterized=True)
        ax.scatter(xs[m], ys[m], s=9, color=CLASS_COLORS[cls], alpha=0.75, linewidth=0, rasterized=True)
        ax.set_title(cls, fontsize=10.5)
    xl, yl = FEATURE_LABELS.get(x, x), FEATURE_LABELS.get(y, y)
    for ax in axes[-1]:
        ax.set_xlabel(("log10 " if logx else "") + xl)
    for ax in axes[:, 0]:
        ax.set_ylabel(("log10 " if logy else "") + yl)
    _suptitle(fig, f"{yl.split(' (')[0]} vs {xl.split(' (')[0].lower()} by class",
              f"Each panel highlights one class (colour) over a sample of all images (grey), {per_class} images per class")
    fig.tight_layout(rect=(0, 0, 1, 0.92))
    return _save(fig, out)


def tsne_small_multiples(coords: np.ndarray, labels: np.ndarray, out, title=None):
    """2-D t-SNE map of deep features; one panel per class highlighted over the others."""
    fig, axes = plt.subplots(2, 3, figsize=(12, 7.4))
    for ax, (k, cls) in zip(axes.ravel(), enumerate(CLASS_NAMES)):
        m = labels == k
        ax.scatter(coords[~m, 0], coords[~m, 1], s=4, color=GREY_MARK, alpha=0.6, linewidth=0, rasterized=True)
        ax.scatter(coords[m, 0], coords[m, 1], s=7, color=CLASS_COLORS[cls], alpha=0.85, linewidth=0, rasterized=True)
        ax.set_title(f"{cls} (n={m.sum():,})", fontsize=10.5)
        ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
        for s in ax.spines.values():
            s.set_edgecolor(GRID)
    _suptitle(fig, title or "t-SNE map of pre-trained EfficientNet-B0 features",
              "Nearby points = images the network considers similar; each panel highlights one class")
    fig.tight_layout(rect=(0, 0, 1, 0.92))
    return _save(fig, out)


def duplicate_histogram(nn_dist: np.ndarray, out, threshold=4):
    """Distribution of the distance from each image to its most similar other image."""
    fig, ax = plt.subplots(figsize=(8, 3.6))
    counts = np.bincount(nn_dist.astype(int), minlength=33)[:33]
    colors = [ACCENT_2 if d <= threshold else ACCENT for d in range(33)]
    ax.bar(np.arange(33), counts, width=0.8, color=colors, zorder=2)
    ax.axvline(threshold + 0.5, color=INK_2, lw=1)
    n_dup = int((nn_dist <= threshold).sum())
    ax.text(threshold + 0.8, counts.max() * 0.95,
            f"<= {threshold}: near-duplicates\n{n_dup:,} images ({n_dup / len(nn_dist):.1%})",
            fontsize=8.5, color=INK_2, va="top")
    ax.set_xlabel("pHash Hamming distance to the most similar other image (0 = identical)")
    ax.set_ylabel("Images")
    _no_grid_x(ax)
    ax.set_title("Near-duplicate analysis with perceptual hashing")
    return _save(fig, out)


def image_pairs(pairs, out, title, max_pairs=6):
    """Show example pairs of images side by side (e.g. near-duplicates)."""
    from PIL import Image

    pairs = pairs[:max_pairs]
    if not pairs:
        return None
    fig, axes = plt.subplots(2, len(pairs), figsize=(2.1 * len(pairs), 4.6), squeeze=False)
    for c, (pa, pb, caption) in enumerate(pairs):
        for r, p in enumerate([pa, pb]):
            ax = axes[r, c]
            with Image.open(p) as im:
                ax.imshow(im.convert("RGB"))
            ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
            for s in ax.spines.values():
                s.set_visible(False)
        axes[1, c].set_xlabel(caption, fontsize=8, color=INK_2)
    _suptitle(fig, title)
    fig.tight_layout(rect=(0, 0, 1, 0.9))
    return _save(fig, out)


def split_distribution(df: pd.DataFrame, out):
    """Images per class in the training, validation and test sets."""
    table = df.groupby(["class_name", "split"]).size().unstack("split").reindex(CLASS_NAMES)[["train", "val", "test"]]
    fig, ax = plt.subplots(figsize=(9, 3.8))
    width = 0.26
    x = np.arange(len(CLASS_NAMES))
    for k, split in enumerate(["train", "val", "test"]):
        bars = ax.bar(x + (k - 1) * (width + 0.02), table[split].to_numpy(), width=width,
                      color=SPLIT_COLORS[split], label=f"{split} ({table[split].sum():,})", zorder=2)
        if split == "test":
            for b in bars:
                ax.text(b.get_x() + b.get_width() / 2, b.get_height(), f"{int(b.get_height())}",
                        ha="center", va="bottom", fontsize=7.5, color=INK_2)
    ax.set_xticks(x, CLASS_NAMES)
    ax.set_ylabel("Images")
    ax.legend(ncol=3, loc="upper left", bbox_to_anchor=(0, 1.02))
    ax.set_ylim(0, table.to_numpy().max() * 1.2)
    _no_grid_x(ax)
    ax.set_title("Stratified group split: images per class and subset", pad=24)
    return _save(fig, out)


def augmentation_examples(paths, augment, out, n_aug=5, seed=42):
    """Original images next to randomly augmented versions (training transformation)."""
    import torch
    from PIL import Image

    torch.manual_seed(seed)
    fig, axes = plt.subplots(len(paths), n_aug + 1, figsize=(1.9 * (n_aug + 1), 2.0 * len(paths)))
    for r, p in enumerate(paths):
        with Image.open(p) as im:
            img = im.convert("RGB")
        views = [img.resize((224, 224))] + [augment(img) for _ in range(n_aug)]
        for c, v in enumerate(views):
            ax = axes[r, c]
            ax.imshow(v); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
            for s in ax.spines.values():
                s.set_visible(False)
            if r == 0:
                ax.set_title("original" if c == 0 else f"augmented #{c}", fontsize=9, loc="center")
    _suptitle(fig, "Data augmentation applied to training images")
    fig.tight_layout(rect=(0, 0, 1, 0.94))
    return _save(fig, out)


# =======================================================================================
# Part 2 - modelling and evaluation
# =======================================================================================
def learning_curves(history: pd.DataFrame, out, title="Learning curves - EfficientNet-B0"):
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.9))
    ep = history["epoch"]
    switch = history.loc[history["phase"] != history["phase"].iloc[0], "epoch"]
    for ax, metric, label in [(axes[0], "loss", "Cross-entropy loss"), (axes[1], "acc", "Accuracy")]:
        ax.plot(ep, history[f"train_{metric}"], color=ACCENT, marker="o", ms=4, label="training")
        ax.plot(ep, history[f"val_{metric}"], color=ACCENT_2, marker="o", ms=4, label="validation")
        if len(switch):
            ax.axvline(switch.iloc[0] - 0.5, color=MUTED, lw=1)
            ax.text(switch.iloc[0] - 0.4, ax.get_ylim()[1], "  fine-tuning starts", fontsize=8,
                    color=MUTED, va="top")
        ax.set_xlabel("Epoch"); ax.set_title(label)
        ax.set_xticks(ep)
    axes[0].legend(loc="upper right")
    _suptitle(fig, title)
    fig.tight_layout(rect=(0, 0, 1, 0.92))
    return _save(fig, out)


def model_comparison(results: pd.DataFrame, out, metric="test_macro_f1", highlight=None):
    """Horizontal bars: test macro-F1 of every model (the chosen model highlighted)."""
    df = results.sort_values(metric)
    fig, ax = plt.subplots(figsize=(9, 0.55 * len(df) + 1.3))
    colors = [ACCENT if m == highlight else GREY_MARK for m in df["model"]]
    ax.barh(df["model"], df[metric], height=0.55, color=colors, zorder=2)
    for i, v in enumerate(df[metric]):
        ax.text(v + 0.005, i, f"{v:.3f}", va="center", fontsize=8.5, color=INK_2)
    ax.set_xlim(0, 1.08)
    ax.set_xlabel("Macro-F1 on the test set")
    _no_grid_y(ax)
    ax.set_title("Model comparison (same train / validation / test split)")
    return _save(fig, out)


def confusion_matrix_plot(cm: np.ndarray, out, title="Confusion matrix - test set"):
    cm_norm = cm / cm.sum(1, keepdims=True)
    fig, ax = plt.subplots(figsize=(6.6, 5.6))
    im = ax.imshow(cm_norm, cmap=SEQ, vmin=0, vmax=1)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, f"{cm_norm[i, j]:.1%}\n({cm[i, j]})", ha="center", va="center", fontsize=8,
                    color="white" if cm_norm[i, j] > 0.6 else INK)
    ax.set_xticks(range(len(CLASS_NAMES)), CLASS_NAMES, rotation=30, ha="right")
    ax.set_yticks(range(len(CLASS_NAMES)), CLASS_NAMES)
    ax.set_xlabel("Predicted class"); ax.set_ylabel("True class")
    ax.grid(False)
    for s in ax.spines.values():
        s.set_visible(False)
    cb = fig.colorbar(im, ax=ax, fraction=0.045, pad=0.03)
    cb.set_label("Share of the true class (row %)", color=INK_2); cb.outline.set_visible(False)
    ax.set_title(title)
    return _save(fig, out)


def per_class_metrics(report: pd.DataFrame, out):
    """Dot plot of precision, recall and F1 for each class."""
    fig, ax = plt.subplots(figsize=(8, 3.9))
    y = np.arange(len(report))
    styles = [("precision", "#2a78d6", "o"), ("recall", "#eb6834", "s"), ("f1", "#1baf7a", "D")]
    lo = min(report[["precision", "recall", "f1"]].min().min(), 0.9) - 0.03
    for k, (col, color, marker) in enumerate(styles):
        ax.scatter(report[col], y + (k - 1) * 0.18, color=color, marker=marker, s=46,
                   edgecolor=SURFACE, linewidth=1.5, zorder=3, label=col if col != "f1" else "F1-score")
    for i, row in report.iterrows():
        ax.text(1.005, i, f"F1 {row.f1:.3f}", va="center", fontsize=8, color=INK_2)
    ax.set_yticks(y, report["class"])
    ax.invert_yaxis()
    ax.set_xlim(max(0.0, lo), 1.05)
    ax.set_xlabel("Score on the test set")
    ax.legend(ncol=3, loc="upper left", bbox_to_anchor=(0, 1.12))
    ax.set_title("Per-class precision, recall and F1", pad=26)
    return _save(fig, out)


def roc_pr_curves(y: np.ndarray, prob: np.ndarray, out):
    """One-vs-rest ROC and precision-recall curves for every class."""
    from sklearn.metrics import auc, average_precision_score, precision_recall_curve, roc_curve

    fig, axes = plt.subplots(1, 3, figsize=(14, 4.3))
    for k, cls in enumerate(CLASS_NAMES):
        yk = (y == k).astype(int)
        if yk.sum() == 0:
            continue
        fpr, tpr, _ = roc_curve(yk, prob[:, k])
        prec, rec, _ = precision_recall_curve(yk, prob[:, k])
        roc_auc = auc(fpr, tpr)
        ap = average_precision_score(yk, prob[:, k])
        for ax in axes[:2]:
            ax.plot(fpr, tpr, color=CLASS_COLORS[cls], lw=1.8, label=f"{cls} (AUC {roc_auc:.3f})")
        axes[2].plot(rec, prec, color=CLASS_COLORS[cls], lw=1.8, label=f"{cls} (AP {ap:.3f})")
    axes[0].plot([0, 1], [0, 1], color=MUTED, lw=1)
    axes[0].set_title("ROC curves (one-vs-rest)")
    axes[0].set_xlabel("False-positive rate"); axes[0].set_ylabel("True-positive rate")
    axes[0].legend(loc="lower right", fontsize=7.5)
    axes[1].set_xlim(0, 0.2); axes[1].set_ylim(0.8, 1.001)
    axes[1].set_title("ROC - zoom on the top-left corner")
    axes[1].set_xlabel("False-positive rate")
    axes[2].set_title("Precision-recall curves")
    axes[2].set_xlabel("Recall"); axes[2].set_ylabel("Precision")
    axes[2].set_ylim(0.5, 1.01)
    axes[2].legend(loc="lower left", fontsize=7.5)
    fig.tight_layout()
    return _save(fig, out)


def reliability_diagram(before: pd.DataFrame, after: pd.DataFrame, ece_before, ece_after, temperature, out,
                        min_count: int = 10):
    """Calibration curve (top) and number of test images per confidence bin (bottom).

    Marker area is proportional to the number of images in the bin; bins with fewer than
    `min_count` images are drawn hollow because their accuracy is very uncertain.
    """
    fig, (ax, axh) = plt.subplots(2, 1, figsize=(6.2, 6.4), sharex=True,
                                  gridspec_kw={"height_ratios": [3.2, 1], "hspace": 0.08})
    ax.plot([0, 1], [0, 1], color=MUTED, lw=1, label="perfect calibration", zorder=1)
    width = 0.1 / 2.6
    for k, (df, color, label) in enumerate([(before, ACCENT_2, f"before scaling (ECE {ece_before:.3f})"),
                                            (after, ACCENT, f"after T = {temperature:.2f} (ECE {ece_after:.3f})")]):
        d = df.dropna(subset=["confidence"])
        ax.plot(d["confidence"], d["accuracy"], color=color, lw=1.4, alpha=0.8, zorder=2, label=label)
        big = d["count"] >= min_count
        sizes = 14 + 220 * np.sqrt(d["count"] / max(d["count"].max(), 1))
        ax.scatter(d.loc[big, "confidence"], d.loc[big, "accuracy"], s=sizes[big], color=color, zorder=3,
                   edgecolor=SURFACE, linewidth=0.8)
        ax.scatter(d.loc[~big, "confidence"], d.loc[~big, "accuracy"], s=sizes[~big], facecolor=SURFACE,
                   edgecolor=color, linewidth=1.2, zorder=3)
        centres = (df["bin_low"] + df["bin_high"]) / 2 + (k - 0.5) * width
        axh.bar(centres, df["count"].clip(lower=0.8), width=width, color=color, zorder=2)
    ax.set_ylabel("Observed accuracy")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.03)
    ax.legend(loc="lower right", fontsize=8)
    ax.text(0.98, 0.25, f"marker area ~ images per bin\nhollow = fewer than {min_count} images",
            transform=ax.transAxes, fontsize=7.5, color=INK_2, va="bottom", ha="right")
    ax.set_title("Reliability diagram (test set)")
    axh.set_yscale("log")
    axh.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}"))
    axh.yaxis.set_minor_formatter(NullFormatter())
    axh.set_ylim(0.8, None)
    axh.set_ylabel("Images")
    axh.set_xlabel("Mean predicted confidence (10 bins)")
    _no_grid_x(axh)
    return _save(fig, out)


def confidence_analysis(prob: np.ndarray, y: np.ndarray, selective: pd.DataFrame, threshold, out):
    conf, correct = prob.max(1), prob.argmax(1) == y
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.9))
    bins = np.linspace(0, 1, 26)
    ax = axes[0]
    ax.hist(conf[correct], bins=bins, color=ACCENT, alpha=0.85, label=f"correct ({correct.sum():,})", zorder=2)
    ax.hist(conf[~correct], bins=bins, color=ACCENT_2, alpha=0.9, label=f"wrong ({(~correct).sum():,})", zorder=3)
    ax.set_yscale("log")
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}"))
    ax.yaxis.set_minor_formatter(NullFormatter())
    ax.set_xlabel("Confidence of the predicted class"); ax.set_ylabel("Images (log scale)")
    ax.legend(loc="upper left")
    ax.set_title("Confidence of correct vs wrong predictions")
    ax = axes[1]
    ax.plot(selective["threshold"], selective["accuracy"], color=ACCENT, marker="o", ms=3.5, label="accuracy of accepted")
    ax.plot(selective["threshold"], selective["coverage"], color=ACCENT_2, marker="s", ms=3.5, label="share accepted (coverage)")
    ax.axvline(threshold, color=INK_2, lw=1)
    row = selective.iloc[(selective["threshold"] - threshold).abs().argmin()]
    # annotation placed in axes coordinates so it always stays inside the plot
    ax.annotate(f"app threshold {threshold:.0%}\naccuracy {row.accuracy:.1%}\ncoverage {row.coverage:.1%}",
                xy=(threshold, 0.5), xycoords=("data", "axes fraction"), xytext=(-8, 0), textcoords="offset points",
                fontsize=8, color=INK_2, va="center", ha="right",
                bbox={"facecolor": SURFACE, "edgecolor": GRID, "boxstyle": "round,pad=0.3"})
    ax.set_xlabel("Confidence threshold"); ax.set_ylabel("Proportion")
    ax.legend(loc="lower left")
    ax.set_title("Selective prediction: rejecting uncertain images")
    fig.tight_layout()
    return _save(fig, out)


def image_gallery(items, out, title, cols=6, subtitle=None):
    """items: list of (image_path, caption, is_correct) shown in a grid."""
    from PIL import Image

    if not items:
        return None
    rows = int(np.ceil(len(items) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(2.05 * cols, 2.6 * rows + 0.7), squeeze=False)
    for ax in axes.ravel():
        ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
        for s in ax.spines.values():
            s.set_visible(False)
    for ax, (path, caption, ok) in zip(axes.ravel(), items):
        with Image.open(path) as im:
            ax.imshow(im.convert("RGB"))
        # caption drawn in axes coordinates so that two-line captions never hide behind the next row
        ax.text(0.5, -0.04, caption, transform=ax.transAxes, ha="center", va="top", fontsize=7.8,
                color=INK if ok else "#b52e2e")
    _suptitle(fig, title, subtitle)
    fig.subplots_adjust(left=0.01, right=0.99, top=0.86 if subtitle else 0.9, bottom=0.07, wspace=0.08, hspace=0.42)
    return _save(fig, out)


def gradcam_grid(images: np.ndarray, cams: np.ndarray, captions, out, cols=2):
    """Pairs of (image, Grad-CAM overlay) with the caption under each pair."""
    n = len(images)
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols * 2, figsize=(2.0 * cols * 2, 2.55 * rows + 0.6), squeeze=False)
    for ax in axes.ravel():
        ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
        for s in ax.spines.values():
            s.set_visible(False)
    for k in range(n):
        r, c = divmod(k, cols)
        axes[r, 2 * c].imshow(images[k])
        axes[r, 2 * c + 1].imshow(images[k])
        axes[r, 2 * c + 1].imshow(cams[k], cmap="jet", alpha=0.45)
        axes[r, 2 * c].text(0.5, -0.05, captions[k], transform=axes[r, 2 * c].transAxes, ha="center", va="top",
                            fontsize=8, color=INK_2)
        axes[r, 2 * c + 1].text(0.5, -0.05, "Grad-CAM", transform=axes[r, 2 * c + 1].transAxes, ha="center",
                                va="top", fontsize=8, color=MUTED)
    for k in range(n, rows * cols):
        r, c = divmod(k, cols)
        axes[r, 2 * c].set_visible(False); axes[r, 2 * c + 1].set_visible(False)
    _suptitle(fig, "Grad-CAM: image regions that drove the prediction",
              "Red = high influence on the predicted class, blue = low influence; last pairs = most confident mistakes")
    fig.subplots_adjust(left=0.01, right=0.99, top=0.9, bottom=0.04, wspace=0.05, hspace=0.25)
    return _save(fig, out)


def robustness_bars(df: pd.DataFrame, out):
    df = df.copy()
    fig, ax = plt.subplots(figsize=(8.5, 0.5 * len(df) + 1.2))
    colors = [ACCENT if c == "original" else "#9ec5f4" for c in df["corruption"]]
    ax.barh(df["corruption"], df["accuracy"], height=0.55, color=colors, zorder=2)
    for i, v in enumerate(df["accuracy"]):
        ax.text(v + 0.005, i, f"{v:.1%}", va="center", fontsize=8.5, color=INK_2)
    ax.invert_yaxis()
    ax.set_xlim(0, 1.08)
    ax.set_xlabel("Accuracy on the test set after the corruption")
    _no_grid_y(ax)
    ax.set_title("Robustness to image corruptions")
    return _save(fig, out)


def provenance_bars(summary: pd.DataFrame, out):
    """Share of each class that matches each candidate source dataset (100% stacked bars)."""
    df = summary.drop(index="total", errors="ignore").reindex(CLASS_NAMES).fillna(0)
    order = [c for c in df.columns if c != "unidentified"] + (["unidentified"] if "unidentified" in df.columns else [])
    df = df[order]
    shares = df.div(df.sum(axis=1), axis=0)
    colors = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"][: len(order) - 1] + [GREY_MARK]
    fig, ax = plt.subplots(figsize=(10, 3.9))
    left = np.zeros(len(shares))
    for col, color in zip(order, colors):
        vals = shares[col].to_numpy()
        ax.barh(shares.index, vals, left=left, height=0.55, color=color, edgecolor=SURFACE, linewidth=2,
                label=col, zorder=2)
        for i, (l, v) in enumerate(zip(left, vals)):
            if v >= 0.08:
                ax.text(l + v / 2, i, f"{v:.0%}", ha="center", va="center", fontsize=8,
                        color="white" if color != GREY_MARK else INK)
        left += vals
    ax.invert_yaxis()
    ax.set_xlim(0, 1)
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:.0%}"))
    ax.set_xlabel("Share of the class's images whose perceptual hash matches the source (distance <= 6)")
    ax.legend(ncol=2, loc="upper left", bbox_to_anchor=(0, -0.22), fontsize=8)
    _no_grid_y(ax)
    ax.set_title("Where the images come from: matches with public source datasets")
    return _save(fig, out)


In [ ]:
%%writefile waste_classifier/provenance.py
"""Where did the images come from? Empirical provenance check.

The Kaggle dataset was assembled from several public garbage datasets, but the
notebook that built it is private. This module downloads candidate source datasets,
computes a perceptual hash (pHash) of each of their images and looks for matching
hashes among our images. An image whose hash is within a small Hamming distance of a
source image is attributed to that source.
"""
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

from .config import CLASS_NAMES
from .image_analysis import _popcount64, hashes_to_uint64

# Candidate sources (all public on Kaggle; cited in the report)
# Older result files used the label "Garbage Classification v2 (Suman, 2024)" for the third source.
OLD_SOURCE_NAMES = {"Garbage Classification v2 (Suman, 2024)": "Garbage Dataset v2 (Kunwar, 2026)"}
SOURCES = [
    {"name": "TrashNet (Thung & Yang, 2016)", "handle": "asdasdasasdas/garbage-classification"},
    {"name": "Garbage Classification 12 classes (Mohamed, 2021)", "handle": "mostafaabla/garbage-classification"},
    {"name": "Garbage Dataset v2 (Kunwar, 2026)", "handle": "sumn2u/garbage-classification-v2"},
]
IMAGE_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def _hash_variants(path: Path) -> list[int]:
    """pHash of the image squashed to a square and of its centre crop."""
    import imagehash

    with Image.open(path) as im:
        im = im.convert("RGB")
        w, h = im.size
        s = min(w, h)
        crop = im.crop(((w - s) // 2, (h - s) // 2, (w - s) // 2 + s, (h - s) // 2 + s))
        return [str(imagehash.phash(im.resize((256, 256)))), str(imagehash.phash(crop.resize((256, 256))))]


def _hash_chunk(paths):
    rows = []
    for p in paths:
        try:
            h1, h2 = _hash_variants(Path(p))
        except Exception:  # unreadable source file
            continue
        rows.append({"path": str(p), "source_class": Path(p).parent.name.lower(), "h_squash": h1, "h_crop": h2})
    return rows


def hash_source(root: Path, max_images: int | None = None, workers: int = 2, log=print) -> pd.DataFrame:
    """Perceptual hashes of every image of a candidate source data set (in parallel)."""
    from concurrent.futures import ProcessPoolExecutor

    files = sorted(str(p) for p in Path(root).rglob("*") if p.suffix.lower() in IMAGE_EXT)
    if max_images:
        files = files[:max_images]
    chunks = [files[i:i + 500] for i in range(0, len(files), 500)]
    rows = []
    with ProcessPoolExecutor(max_workers=max(1, workers)) as pool:
        for part in pool.map(_hash_chunk, chunks):
            rows += part
    log(f"    hashed {len(rows):,} images")
    return pd.DataFrame(rows)


def min_distance(query: np.ndarray, ref: np.ndarray, block: int = 512) -> tuple[np.ndarray, np.ndarray]:
    """For every query hash: smallest Hamming distance to any reference hash and its index."""
    best = np.full(len(query), 64, dtype=np.int64)
    arg = np.zeros(len(query), dtype=np.int64)
    for start in range(0, len(query), block):
        d = _popcount64(np.bitwise_xor(query[start:start + block, None], ref[None, :])).astype(np.int64)
        best[start:start + block] = d.min(axis=1)
        arg[start:start + block] = d.argmin(axis=1)
    return best, arg


def check_provenance(stats: pd.DataFrame, out_dir, threshold: int = 6, workers: int = 2, log=print) -> pd.DataFrame:
    """Attribute every image of `stats` (needs 'phash' and 'class_name') to a source dataset."""
    import kagglehub

    out_dir = Path(out_dir)
    ours = stats[stats["readable"].fillna(False).astype(bool)].reset_index(drop=True)
    q = hashes_to_uint64(ours["phash"])
    result = ours[["rel_path", "class_name"]].copy()
    result["source"], result["distance"], result["source_class"] = "unidentified", 64, ""
    for src in SOURCES:
        try:
            log(f"  downloading {src['handle']} ...")
            root = Path(kagglehub.dataset_download(src["handle"]))
            ref = hash_source(root, workers=workers, log=log)
        except Exception as exc:
            log(f"  could not use {src['handle']}: {exc}")
            continue
        ref_hashes = np.concatenate([hashes_to_uint64(ref["h_squash"]), hashes_to_uint64(ref["h_crop"])])
        ref_class = np.concatenate([ref["source_class"].to_numpy(), ref["source_class"].to_numpy()])
        d, a = min_distance(q, ref_hashes)
        better = (d <= threshold) & (d < result["distance"].to_numpy())
        result.loc[better, "source"] = src["name"]
        result.loc[better, "distance"] = d[better]
        result.loc[better, "source_class"] = ref_class[a[better]]
        log(f"  {src['name']}: {len(ref):,} source images, {(d <= threshold).sum():,} of our images match")
    result.to_csv(out_dir / "provenance_per_image.csv", index=False)
    summary = result.groupby(["class_name", "source"]).size().unstack("source", fill_value=0).reindex(CLASS_NAMES)
    summary.loc["total"] = summary.sum()
    summary.to_csv(out_dir / "provenance_summary.csv")
    return result


In [ ]:
%%writefile waste_classifier/pipeline.py
"""End-to-end pipeline, split into the steps of the report.

Each step can be called on its own (the Colab notebook runs them one cell at a time)
or all together with `run_all` (scripts/run_pipeline.py). Every step writes its
figures to <out>/figures and its numbers to <out>/metrics, so the report can be
rebuilt from the saved outputs.
"""
from __future__ import annotations

import shutil
import time
import zipfile
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd

from . import plots
from .config import (CLASS_NAMES, IMAGENET_MEAN, IMAGENET_STD, BaselineConfig, SplitConfig,
                     TrainConfig)
from .data import (augmentation_preview, build_index, build_transforms, compare_with_metadata,
                   extract_zip, find_dataset_root, find_metadata_csv, load_rgb, make_dataset,
                   make_loader, make_splits)
from .image_analysis import (NUMERIC_FEATURES, analyse_dataset, duplicate_groups, hashes_to_uint64,
                             near_duplicates)
from .utils import environment_info, save_json, set_seed

KAGGLE_HANDLE = "zlatan599/garbage-dataset-classification"
APP_THRESHOLD = 0.60  # confidence below which the app reports "uncertain"


@dataclass
class Workspace:
    """Output folders of one pipeline run."""

    out_dir: Path

    def __post_init__(self):
        self.out_dir = Path(self.out_dir)
        for d in [self.fig_dir, self.metrics_dir, self.model_dir, self.cache_dir, self.sample_dir]:
            d.mkdir(parents=True, exist_ok=True)

    @property
    def fig_dir(self):
        return self.out_dir / "figures"

    @property
    def metrics_dir(self):
        return self.out_dir / "metrics"

    @property
    def model_dir(self):
        return self.out_dir / "model"

    @property
    def cache_dir(self):
        return self.out_dir / "cache"

    @property
    def sample_dir(self):
        return self.out_dir / "sample_images"

    def fig(self, name):
        return self.fig_dir / name


@dataclass
class Context:
    """Objects shared between the steps."""

    root: Path | None = None
    index: pd.DataFrame | None = None
    stats: pd.DataFrame | None = None
    pairs: pd.DataFrame | None = None
    mean_images: np.ndarray | None = None
    summary: dict = field(default_factory=dict)
    data: pd.DataFrame | None = None          # readable images + split column
    embeddings: np.ndarray | None = None
    baselines: pd.DataFrame | None = None
    model: object = None
    history: pd.DataFrame | None = None
    temperature: float = 1.0
    test_metrics: dict = field(default_factory=dict)
    timings: dict = field(default_factory=dict)


# =======================================================================================
# Step 0 - get the data
# =======================================================================================
def acquire_data(zip_path=None, data_dir=None, extract_to="/content/data", use_kagglehub=True, log=print) -> Path:
    """Return the dataset root (the folder that contains one sub-folder per class).

    Priority: an already extracted folder -> a zip file (e.g. on Google Drive) -> kagglehub.
    """
    if data_dir and Path(data_dir).exists():
        return find_dataset_root(data_dir)
    if zip_path and Path(zip_path).exists():
        log(f"Extracting {zip_path} ...")
        return find_dataset_root(extract_zip(zip_path, extract_to))
    if use_kagglehub:
        import kagglehub

        log(f"Downloading {KAGGLE_HANDLE} with kagglehub ...")
        return find_dataset_root(kagglehub.dataset_download(KAGGLE_HANDLE))
    raise FileNotFoundError("Dataset not found: give zip_path, data_dir or enable kagglehub")


# =======================================================================================
# Step 1.1 - description of the data set (counts, formats, quality, duplicates)
# =======================================================================================
def data_understanding(root, ws: Workspace, split_cfg=SplitConfig(), workers=2, log=print) -> Context:
    t0 = time.time()
    ctx = Context(root=Path(root))
    index = build_index(root)
    other_files = index.loc[~index["is_image_file"], "rel_path"].tolist()
    index = index[index["is_image_file"]].reset_index(drop=True)
    log(f"Found {len(index):,} image files in {index['class_name'].nunique()} class folders "
        f"({len(other_files)} other files)")

    stats, mean_imgs = analyse_dataset(index, len(CLASS_NAMES), workers=workers, log=log)
    ok = stats["readable"].fillna(False).to_numpy(dtype=bool)
    log(f"Readable images: {ok.sum():,} / {len(stats):,}")

    # ---- duplicates ------------------------------------------------------------------
    stats["md5_dup_count"] = stats.groupby("md5")["md5"].transform("size")
    readable_idx = np.flatnonzero(ok)
    hashes = hashes_to_uint64(stats.loc[ok, "phash"])
    pairs_r, nn_r = near_duplicates(hashes, threshold=max(8, split_cfg.dup_threshold))
    pairs = pairs_r.assign(i=readable_idx[pairs_r["i"]], j=readable_idx[pairs_r["j"]])
    stats["nn_phash_distance"] = np.nan
    stats.loc[ok, "nn_phash_distance"] = nn_r
    close = pairs[pairs["distance"] <= split_cfg.dup_threshold]
    stats["dup_group"] = duplicate_groups(len(stats), close)
    lab = stats["label"].to_numpy()
    pairs["same_class"] = lab[pairs["i"]] == lab[pairs["j"]]

    summary = {
        "dataset_root": str(root),
        "n_images": int(len(stats)),
        "n_readable": int(ok.sum()),
        "n_classes": int(stats["class_name"].nunique()),
        "class_counts": stats["class_name"].value_counts().reindex(CLASS_NAMES).fillna(0).astype(int).to_dict(),
        "formats": stats["format"].value_counts(dropna=False).to_dict(),
        "modes": stats["mode"].value_counts(dropna=False).to_dict(),
        "extensions": stats["ext"].value_counts().to_dict(),
        "resolutions": {f"{int(w)}x{int(h)}": int(n) for (w, h), n in
                        stats.groupby(["width", "height"]).size().sort_values(ascending=False).head(10).items()},
        "file_size_kb": stats["file_size_kb"].describe().round(2).to_dict(),
        "total_size_mb": float(stats["file_size_kb"].sum() / 1024),
        "other_files": other_files,
        "unreadable_files": stats.loc[~ok, ["rel_path", "error"]].to_dict("records"),
        "exact_duplicate_files": int((stats["md5_dup_count"] > 1).sum()),
        "exact_duplicate_groups": int(stats.loc[stats["md5_dup_count"] > 1, "md5"].nunique()),
        "near_duplicate_pairs": {f"<={t}": int((pairs["distance"] <= t).sum()) for t in [0, 2, 4, 6, 8]},
        "images_with_near_duplicate": {f"<={t}": int((stats["nn_phash_distance"] <= t).sum()) for t in [0, 2, 4, 6, 8]},
        "cross_class_pairs": {f"<={t}": int(((pairs["distance"] <= t) & ~pairs["same_class"]).sum()) for t in [0, 2, 4]},
        "dup_threshold": split_cfg.dup_threshold,
        "n_dup_groups_multi": int((stats["dup_group"].value_counts() > 1).sum()),
    }
    counts = pd.Series(summary["class_counts"])
    shares = counts / counts.sum()
    summary["imbalance_ratio_max_min"] = float(counts.max() / max(counts.min(), 1))
    summary["class_share_min_max"] = [float(shares.min()), float(shares.max())]
    summary["shannon_evenness"] = float(-(shares * np.log(shares)).sum() / np.log(len(shares)))
    # Class balance once byte-identical copies are counted only once (the "true" number of photos).
    unique = stats[ok].groupby("class_name")["md5"].nunique().reindex(CLASS_NAMES).fillna(0).astype(int)
    summary["unique_images_per_class"] = unique.to_dict()
    summary["unique_images_total"] = int(stats.loc[ok, "md5"].nunique())
    summary["imbalance_ratio_unique"] = float(unique.max() / max(unique.min(), 1))
    summary["duplicate_share_by_class"] = {c: round(float(1 - unique[c] / counts[c]), 4) if counts[c] else 0.0
                                           for c in CLASS_NAMES}
    meta = find_metadata_csv(root)
    if meta is not None:
        try:
            summary["metadata_check"] = compare_with_metadata(index, meta)
        except Exception as exc:  # metadata is only a cross-check
            summary["metadata_check"] = {"error": str(exc)}
    feat_summary = stats.groupby("class_name")[NUMERIC_FEATURES].agg(["mean", "std", "median"]).round(4)
    feat_summary.to_csv(ws.metrics_dir / "image_statistics_by_class.csv")
    stats[NUMERIC_FEATURES].describe().T.round(4).to_csv(ws.metrics_dir / "image_statistics_overall.csv")

    stats.to_pickle(ws.cache_dir / "image_stats.pkl")
    light_cols = [c for c in stats.columns if "_hist_" not in c and not c.startswith("hue_")]
    stats[light_cols].drop(columns=["path"]).to_csv(ws.metrics_dir / "image_stats.csv", index=False)
    pairs.to_csv(ws.metrics_dir / "near_duplicate_pairs.csv", index=False)
    np.save(ws.cache_dir / "mean_images.npy", mean_imgs)
    save_json(summary, ws.metrics_dir / "dataset_summary.json")

    ctx.index, ctx.stats, ctx.pairs, ctx.mean_images, ctx.summary = index, stats, pairs, mean_imgs, summary
    ctx.timings["data_understanding_s"] = time.time() - t0
    log(f"Class counts: {summary['class_counts']}")
    log(f"Exact duplicate files: {summary['exact_duplicate_files']}, near-duplicate pairs "
        f"(<= {split_cfg.dup_threshold}): {summary['near_duplicate_pairs'].get(f'<={split_cfg.dup_threshold}', 'n/a')}")
    return ctx


# =======================================================================================
# Step 1.2 - visualisation
# =======================================================================================
def visualise(ctx: Context, ws: Workspace, log=print) -> list:
    t0 = time.time()
    s = ctx.stats[ctx.stats["readable"].fillna(False).astype(bool)]
    figs = [
        plots.class_distribution(ctx.stats["class_name"].value_counts(), ws.fig("fig01_class_distribution.png")),
        plots.duplicates_by_class(s, ws.fig("fig01b_duplicates_by_class.png")),
        plots.sample_grid(s, ws.fig("fig02_sample_images.png")),
        plots.file_properties(ctx.stats, ws.fig("fig03_file_properties.png")),
        plots.feature_boxplots(s, ["brightness", "contrast", "saturation", "colourfulness", "sharpness",
                                   "entropy", "edge_density", "file_size_kb"],
                               ws.fig("fig04_image_statistics_by_class.png")),
        plots.white_background_share(s, ws.fig("fig04b_white_background.png")),
        plots.channel_histograms(s, ws.fig("fig05_rgb_histograms.png")),
        plots.hue_heatmap(s, ws.fig("fig06_hue_heatmap.png")),
        plots.mean_images(ctx.mean_images, ws.fig("fig07_mean_images.png")),
        plots.correlation_heatmap(s, NUMERIC_FEATURES, ws.fig("fig08_correlation_heatmap.png")),
        plots.scatter_by_class(s, "brightness", "saturation", ws.fig("fig09_brightness_vs_saturation.png")),
        plots.duplicate_histogram(s["nn_phash_distance"].to_numpy(), ws.fig("fig10_near_duplicates.png"),
                                  threshold=ctx.summary["dup_threshold"]),
    ]
    # Example near-duplicate pairs (same class first, then pairs with different labels).
    st, p = ctx.stats, ctx.pairs
    examples = []
    for _, row in p[p["distance"] <= ctx.summary["dup_threshold"]].sort_values("distance").head(4).iterrows():
        a, b = st.iloc[int(row.i)], st.iloc[int(row.j)]
        examples.append((a.path, b.path, f"{a.class_name} / {b.class_name}\ndistance {int(row.distance)}"))
    for _, row in p[(~p["same_class"]) & (p["distance"] <= ctx.summary["dup_threshold"])].head(3).iterrows():
        a, b = st.iloc[int(row.i)], st.iloc[int(row.j)]
        examples.append((a.path, b.path, f"{a.class_name} / {b.class_name}\ndistance {int(row.distance)}"))
    if examples:
        figs.append(plots.image_pairs(examples, ws.fig("fig11_duplicate_examples.png"),
                                      "Examples of near-duplicate pairs (top and bottom image of each column)"))
    ctx.timings["visualise_s"] = time.time() - t0
    log(f"Saved {len([f for f in figs if f])} figures to {ws.fig_dir}")
    return [f for f in figs if f]


def provenance(ctx: Context, ws: Workspace, workers=2, log=print):
    """Which public data sets do the images come from? (perceptual-hash matching)"""
    from .provenance import check_provenance

    t0 = time.time()
    check_provenance(ctx.stats, ws.metrics_dir, workers=workers, log=log)
    summary = pd.read_csv(ws.metrics_dir / "provenance_summary.csv", index_col=0)
    plots.provenance_bars(summary, ws.fig("fig25_provenance.png"))
    ctx.timings["provenance_s"] = time.time() - t0
    return summary


# =======================================================================================
# Step 1.2 (cont.) - deep features: t-SNE map of class similarity
# =======================================================================================
def deep_embeddings(ctx: Context, ws: Workspace, device, batch_size=128, workers=2, log=print) -> np.ndarray:
    """1,280-d EfficientNet-B0 (ImageNet) embedding of every readable image (cached)."""
    from .models import create_feature_extractor, extract_embeddings

    cache = ws.cache_dir / "embeddings_imagenet.npz"
    readable = ctx.stats[ctx.stats["readable"].fillna(False).astype(bool)]
    if cache.exists():
        z = np.load(cache, allow_pickle=True)
        if len(z["paths"]) == len(readable):
            ctx.embeddings = z["emb"]
            return ctx.embeddings
    t0 = time.time()
    ds = make_dataset(readable["path"], readable["label"], build_transforms(224, train=False))
    loader = make_loader(ds, batch_size, shuffle=False, num_workers=workers, device_type=device.type)
    emb, _ = extract_embeddings(create_feature_extractor(pretrained=True), loader, device)
    np.savez_compressed(cache, emb=emb.astype(np.float16), paths=readable["path"].to_numpy())
    ctx.embeddings = emb.astype(np.float32)
    ctx.timings["embeddings_s"] = time.time() - t0
    log(f"Embeddings: {emb.shape} in {time.time() - t0:.0f}s")
    return ctx.embeddings


def tsne_map(ctx: Context, ws: Workspace, per_class=500, seed=42, log=print):
    from sklearn.decomposition import PCA
    from sklearn.manifold import TSNE

    readable = ctx.stats[ctx.stats["readable"].fillna(False).astype(bool)].reset_index(drop=True)
    rng = np.random.default_rng(seed)
    idx = np.concatenate([rng.choice(np.flatnonzero(readable["label"] == k),
                                     size=min(per_class, int((readable["label"] == k).sum())), replace=False)
                          for k in range(len(CLASS_NAMES)) if (readable["label"] == k).any()])
    x = PCA(n_components=min(50, len(idx) - 1), random_state=seed).fit_transform(ctx.embeddings[idx])
    coords = TSNE(n_components=2, perplexity=min(35, max(5, len(idx) // 10)), init="pca",
                  learning_rate="auto", random_state=seed).fit_transform(x)
    pd.DataFrame({"x": coords[:, 0], "y": coords[:, 1], "label": readable["label"].to_numpy()[idx]}) \
        .to_csv(ws.metrics_dir / "tsne_coordinates.csv", index=False)
    return plots.tsne_small_multiples(coords, readable["label"].to_numpy()[idx], ws.fig("fig12_tsne_embeddings.png"))


# =======================================================================================
# Step 2.2 - data transformation: split + augmentation
# =======================================================================================
def split_data(ctx: Context, ws: Workspace, split_cfg=SplitConfig(), log=print) -> pd.DataFrame:
    data = ctx.stats[ctx.stats["readable"].fillna(False).astype(bool)].reset_index(drop=True)
    # Exact duplicates inside the same class are kept once (they add no information).
    before = len(data)
    data = data.drop_duplicates(subset=["md5", "label"]).reset_index(drop=True)
    removed_exact = before - len(data)
    # Identical files with *different* labels are ambiguous -> removed completely.
    conflict = data.groupby("md5")["label"].transform("nunique") > 1
    removed_conflicts = int(conflict.sum())
    data = data[~conflict].reset_index(drop=True)
    data["split"] = make_splits(data["label"], data["dup_group"], split_cfg.test_size, split_cfg.val_size, split_cfg.seed)
    ctx.data = data
    data[["rel_path", "class_name", "label", "split", "dup_group"]].to_csv(ws.metrics_dir / "splits.csv", index=False)
    table = data.groupby(["class_name", "split"]).size().unstack("split").reindex(CLASS_NAMES)[["train", "val", "test"]]
    table.loc["total"] = table.sum()
    table.to_csv(ws.metrics_dir / "split_counts.csv")
    ctx.summary.update(removed_exact_duplicates=removed_exact, removed_label_conflicts=removed_conflicts,
                       split_counts=table.to_dict())
    save_json(ctx.summary, ws.metrics_dir / "dataset_summary.json")
    # leakage check: no duplicate group in two splits
    leak = data.groupby("dup_group")["split"].nunique().gt(1).sum()
    log(f"Removed {removed_exact} exact duplicates and {removed_conflicts} conflicting files; "
        f"groups spanning several splits: {leak}")
    log(table.to_string())
    plots.split_distribution(data, ws.fig("fig13_split_distribution.png"))
    rng = np.random.default_rng(split_cfg.seed)
    demo = [data.loc[data["label"] == k, "path"].sample(1, random_state=int(rng.integers(1e9))).iloc[0]
            for k in [0, 1, 4] if (data["label"] == k).any()]
    plots.augmentation_examples(demo, augmentation_preview(224), ws.fig("fig14_augmentation_examples.png"))
    return data


def _subset(ctx, split):
    d = ctx.data
    return d[d["split"] == split]


def _loaders(ctx, img_size, batch_size, workers, device, train_aug=True):
    tr, va, te = (_subset(ctx, s) for s in ["train", "val", "test"])
    ds_tr = make_dataset(tr["path"], tr["label"], build_transforms(img_size, train=train_aug))
    ds_va = make_dataset(va["path"], va["label"], build_transforms(img_size, train=False))
    ds_te = make_dataset(te["path"], te["label"], build_transforms(img_size, train=False))
    return (make_loader(ds_tr, batch_size, True, workers, device.type),
            make_loader(ds_va, batch_size, False, workers, device.type),
            make_loader(ds_te, batch_size, False, workers, device.type))


# =======================================================================================
# Step 2.1 / 2.4 - baseline models for comparison
# =======================================================================================
def _scores(y, pred, prefix):
    from sklearn.metrics import accuracy_score, f1_score

    return {f"{prefix}_accuracy": accuracy_score(y, pred), f"{prefix}_macro_f1": f1_score(y, pred, average="macro")}


def baselines(ctx: Context, ws: Workspace, device, cfg=BaselineConfig(), run_cnn=True, workers=2, log=print) -> pd.DataFrame:
    from sklearn.decomposition import PCA
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.linear_model import LogisticRegression
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.svm import SVC

    from .features import extract_features

    t0 = time.time()
    data = ctx.data
    cache = ws.cache_dir / "handcrafted_features.npy"
    if cache.exists() and np.load(cache).shape[0] == len(data):
        X = np.load(cache)
    else:
        log("Extracting hand-crafted features (colour histogram + HOG + LBP) ...")
        X = extract_features(data["path"], size=cfg.handcrafted_size, workers=workers)
        np.save(cache, X)
    y = data["label"].to_numpy()
    m = {s: (data["split"] == s).to_numpy() for s in ["train", "val", "test"]}
    log(f"Feature matrix {X.shape}")

    n_comp = min(cfg.pca_components, X.shape[1], int(m["train"].sum()) - 1)
    models = {
        "k-NN (k=5) + HOG/colour/LBP": make_pipeline(StandardScaler(), PCA(n_comp, random_state=cfg.seed),
                                                     KNeighborsClassifier(5)),
        "Logistic regression + HOG/colour/LBP": make_pipeline(StandardScaler(), PCA(n_comp, random_state=cfg.seed),
                                                              LogisticRegression(max_iter=3000, C=0.5)),
        "Random forest + HOG/colour/LBP": RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=cfg.seed),
        "SVM (RBF) + HOG/colour/LBP": make_pipeline(StandardScaler(), PCA(n_comp, random_state=cfg.seed),
                                                    SVC(C=10, gamma="scale")),
    }
    rows = []
    for name, clf in models.items():
        start = time.time()
        clf.fit(X[m["train"]], y[m["train"]])
        fit_s = time.time() - start
        row = {"model": name, "family": "classical ML", "fit_seconds": fit_s}
        row.update(_scores(y[m["val"]], clf.predict(X[m["val"]]), "val"))
        row.update(_scores(y[m["test"]], clf.predict(X[m["test"]]), "test"))
        rows.append(row)
        log(f"  {name}: val acc {row['val_accuracy']:.3f}, test acc {row['test_accuracy']:.3f} ({fit_s:.0f}s)")

    # Linear probe: frozen ImageNet features + logistic regression (transfer learning without fine-tuning)
    if ctx.embeddings is not None:
        readable = ctx.stats[ctx.stats["readable"].fillna(False).astype(bool)].reset_index(drop=True)
        pos = pd.Series(np.arange(len(readable)), index=readable["path"])
        E = ctx.embeddings[pos.loc[data["path"]].to_numpy()]
        start = time.time()
        probe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, C=0.5))
        probe.fit(E[m["train"]], y[m["train"]])
        row = {"model": "Frozen EfficientNet-B0 + logistic regression", "family": "transfer learning (frozen)",
               "fit_seconds": time.time() - start}
        row.update(_scores(y[m["val"]], probe.predict(E[m["val"]]), "val"))
        row.update(_scores(y[m["test"]], probe.predict(E[m["test"]]), "test"))
        rows.append(row)
        log(f"  linear probe: val acc {row['val_accuracy']:.3f}, test acc {row['test_accuracy']:.3f}")

    if run_cnn:
        import torch

        from .models import SimpleCNN, count_parameters
        from .train import fit_from_scratch, predict_logits

        set_seed(cfg.seed)
        tr_loader, va_loader, te_loader = _loaders(ctx, cfg.cnn_img_size, cfg.cnn_batch_size, workers, device)
        cnn = SimpleCNN(len(CLASS_NAMES))
        log(f"Small CNN from scratch: {count_parameters(cnn)[0]:,} parameters, {cfg.cnn_epochs} epochs max")
        start = time.time()
        cnn, hist, best_ep = fit_from_scratch(cnn, tr_loader, va_loader, cfg.cnn_epochs, cfg.cnn_lr, device, log=log)
        hist.to_csv(ws.metrics_dir / "history_small_cnn.csv", index=False)
        va_logits, va_y = predict_logits(cnn, va_loader, device)
        te_logits, te_y = predict_logits(cnn, te_loader, device)
        row = {"model": "Small CNN trained from scratch", "family": "deep learning (scratch)",
               "fit_seconds": time.time() - start}
        row.update(_scores(va_y, va_logits.argmax(1), "val"))
        row.update(_scores(te_y, te_logits.argmax(1), "test"))
        rows.append(row)
        plots.learning_curves(hist, ws.fig("fig15b_learning_curves_small_cnn.png"), "Learning curves - small CNN from scratch")
        del cnn
        if device.type == "cuda":
            torch.cuda.empty_cache()

    ctx.baselines = pd.DataFrame(rows)
    ctx.baselines.to_csv(ws.metrics_dir / "baseline_results.csv", index=False)
    ctx.timings["baselines_s"] = time.time() - t0
    return ctx.baselines


# =======================================================================================
# Step 2.3 - train the final model
# =======================================================================================
def train_final(ctx: Context, ws: Workspace, device, cfg=TrainConfig(), workers=2, log=print):
    import torch

    from .models import create_efficientnet, count_parameters
    from .train import fit_transfer_learning

    t0 = time.time()
    set_seed(cfg.seed)
    torch.backends.cudnn.benchmark = True   # fastest convolution algorithms for a fixed input size
    tr_loader, va_loader, _ = _loaders(ctx, cfg.img_size, cfg.batch_size, workers, device)
    model = create_efficientnet(len(CLASS_NAMES), pretrained=True, drop_rate=cfg.drop_rate,
                                drop_path_rate=cfg.drop_path_rate)
    total, _ = count_parameters(model)
    log(f"EfficientNet-B0: {total:,} parameters; train {len(tr_loader.dataset):,} / val {len(va_loader.dataset):,} images")
    model, history, best_epoch = fit_transfer_learning(model, tr_loader, va_loader, cfg, device, log=log)
    history.to_csv(ws.metrics_dir / "history_efficientnet_b0.csv", index=False)
    torch.save(model.state_dict(), ws.model_dir / "efficientnet_b0_best.pt")
    save_json({**cfg.to_dict(), "best_epoch": best_epoch, "train_seconds": time.time() - t0,
               "n_parameters": total}, ws.metrics_dir / "training_config.json")
    plots.learning_curves(history, ws.fig("fig15_learning_curves.png"))
    ctx.model, ctx.history = model, history
    ctx.timings["train_final_s"] = time.time() - t0
    return model, history


# =======================================================================================
# Step 2.4 - evaluation of the final model
# =======================================================================================
def evaluate_final(ctx: Context, ws: Workspace, device, cfg=TrainConfig(), workers=2, threshold=APP_THRESHOLD,
                   robustness_max=1200, log=print) -> dict:
    import torch

    from . import evaluate as ev
    from .explain import GradCAM, denormalise
    from .train import predict_logits

    t0 = time.time()
    model = ctx.model.to(device).eval()
    _, va_loader, te_loader = _loaders(ctx, cfg.img_size, cfg.batch_size, workers, device)
    va_logits, va_y = predict_logits(model, va_loader, device)
    te_logits, te_y = predict_logits(model, te_loader, device)
    T = ev.fit_temperature(va_logits, va_y)
    prob_raw, prob = ev.softmax(te_logits), ev.softmax(te_logits, T)
    ctx.temperature = T

    metrics = ev.classification_metrics(te_y, prob, CLASS_NAMES)
    metrics["ece_before_temperature"] = ev.expected_calibration_error(prob_raw, te_y)
    metrics["log_loss_before_temperature"] = ev.classification_metrics(te_y, prob_raw, CLASS_NAMES)["log_loss"]
    metrics["temperature"] = T
    metrics.update(ev.bootstrap_ci(te_y, prob.argmax(1)))
    val_metrics = ev.classification_metrics(va_y, ev.softmax(va_logits, T), CLASS_NAMES)
    per_class = ev.per_class_report(te_y, prob, CLASS_NAMES)
    cm = ev.confusion(te_y, prob, len(CLASS_NAMES))
    confused = ev.most_confused_pairs(cm, CLASS_NAMES)
    selective = ev.selective_prediction(prob, te_y)
    row_t = selective.iloc[(selective["threshold"] - threshold).abs().argmin()]
    metrics["app_threshold"] = threshold
    metrics["accuracy_above_threshold"] = float(row_t["accuracy"])
    metrics["coverage_above_threshold"] = float(row_t["coverage"])

    test = _subset(ctx, "test").reset_index(drop=True)
    pred_df = test[["rel_path", "class_name", "label"]].copy()
    pred_df["pred_label"] = prob.argmax(1)
    pred_df["pred_class"] = [CLASS_NAMES[k] for k in pred_df["pred_label"]]
    pred_df["confidence"] = prob.max(1)
    for k, c in enumerate(CLASS_NAMES):
        pred_df[f"p_{c}"] = prob[:, k]
    pred_df.to_csv(ws.metrics_dir / "test_predictions.csv", index=False)
    per_class.to_csv(ws.metrics_dir / "per_class_metrics.csv", index=False)
    pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(ws.metrics_dir / "confusion_matrix.csv")
    confused.to_csv(ws.metrics_dir / "most_confused_pairs.csv", index=False)
    selective.to_csv(ws.metrics_dir / "selective_prediction.csv", index=False)

    # ---- figures -------------------------------------------------------------------------
    plots.confusion_matrix_plot(cm, ws.fig("fig16_confusion_matrix.png"))
    plots.per_class_metrics(per_class, ws.fig("fig17_per_class_metrics.png"))
    plots.roc_pr_curves(te_y, prob, ws.fig("fig18_roc_pr_curves.png"))
    plots.reliability_diagram(ev.reliability_table(prob_raw, te_y), ev.reliability_table(prob, te_y),
                              metrics["ece_before_temperature"], metrics["ece"], T, ws.fig("fig19_reliability_diagram.png"))
    plots.confidence_analysis(prob, te_y, selective, threshold, ws.fig("fig20_confidence_analysis.png"))
    wrong = pred_df[pred_df["pred_label"] != pred_df["label"]].sort_values("confidence", ascending=False)
    items = [(test.loc[i, "path"], f"true: {r.class_name}\npred: {r.pred_class} ({r.confidence:.0%})", False)
             for i, r in wrong.head(18).iterrows()]
    plots.image_gallery(items, ws.fig("fig21_misclassified_examples.png"),
                        f"Misclassified test images ({len(wrong)} of {len(pred_df):,})",
                        subtitle="Sorted by the model's confidence in the wrong answer (most confident first)")

    # Grad-CAM on one correct image per class + the two most confident mistakes
    cam = GradCAM(model)
    rng = np.random.default_rng(cfg.seed)
    picks = []
    for k in range(len(CLASS_NAMES)):
        ok_idx = pred_df.index[(pred_df["label"] == k) & (pred_df["pred_label"] == k)]
        if len(ok_idx):
            picks.append(int(rng.choice(ok_idx)))
    picks += [int(i) for i in wrong.index[:2]]
    tf = build_transforms(cfg.img_size, train=False)
    batch = torch.stack([tf(load_rgb(test.loc[i, "path"])) for i in picks]).to(device)
    heat, pred_k = cam(batch)
    cam.remove()
    captions = [f"true {test.loc[i, 'class_name']} / pred {CLASS_NAMES[p]}" for i, p in zip(picks, pred_k)]
    plots.gradcam_grid(denormalise(batch, IMAGENET_MEAN, IMAGENET_STD), heat, captions, ws.fig("fig22_gradcam.png"))

    # Robustness to corruptions (subset of the test set)
    rob_idx = pd.concat([d.sample(min(len(d), robustness_max // len(CLASS_NAMES)), random_state=cfg.seed)
                         for _, d in test.groupby("label")]).index
    rows = []
    for name, fn in ev.corruption_transforms().items():
        preds = []
        for start in range(0, len(rob_idx), cfg.batch_size):
            chunk = rob_idx[start:start + cfg.batch_size]
            xb = torch.stack([tf(fn(load_rgb(test.loc[i, "path"]))) for i in chunk]).to(device)
            with torch.inference_mode():
                preds.append(model(xb).argmax(1).cpu().numpy())
        acc = float((np.concatenate(preds) == test.loc[rob_idx, "label"].to_numpy()).mean())
        rows.append({"corruption": name, "accuracy": acc})
    robustness = pd.DataFrame(rows)
    robustness.to_csv(ws.metrics_dir / "robustness.csv", index=False)
    plots.robustness_bars(robustness, ws.fig("fig23_robustness.png"))

    # Comparison with the baselines
    final_row = {"model": "EfficientNet-B0 fine-tuned (final)", "family": "transfer learning (fine-tuned)",
                 "val_accuracy": val_metrics["accuracy"], "val_macro_f1": val_metrics["macro_f1"],
                 "test_accuracy": metrics["accuracy"], "test_macro_f1": metrics["macro_f1"],
                 "fit_seconds": ctx.timings.get("train_final_s", np.nan)}
    comparison = pd.concat([ctx.baselines if ctx.baselines is not None else pd.DataFrame(),
                            pd.DataFrame([final_row])], ignore_index=True)
    comparison.to_csv(ws.metrics_dir / "model_comparison.csv", index=False)
    plots.model_comparison(comparison, ws.fig("fig24_model_comparison.png"), highlight=final_row["model"])

    metrics["val"] = val_metrics
    metrics["most_confused"] = confused.to_dict("records")
    save_json(metrics, ws.metrics_dir / "test_metrics.json")
    ctx.test_metrics = metrics
    ctx.timings["evaluate_s"] = time.time() - t0
    log(f"Test accuracy {metrics['accuracy']:.4f} (95% CI {metrics['accuracy_ci'][0]:.4f}-{metrics['accuracy_ci'][1]:.4f}), "
        f"macro-F1 {metrics['macro_f1']:.4f}, temperature {T:.2f}")
    log(per_class.round(4).to_string(index=False))
    return metrics


# =======================================================================================
# Step 3 - export the model for the application
# =======================================================================================
def export_model(ctx: Context, ws: Workspace, cfg=TrainConfig(), log=print) -> dict:
    import torch

    from . import evaluate as ev
    from .export import benchmark_latency, compress_weights_fp16, export_onnx, onnx_logits

    import gc

    t0 = time.time()
    gc.collect()                    # release memory held by earlier steps (data loaders, features)
    model = ctx.model.eval().cpu()
    fp32 = export_onnx(model, ws.cache_dir / "waste_classifier_fp32.onnx", cfg.img_size)
    final = compress_weights_fp16(fp32, ws.model_dir / "waste_classifier.onnx")
    test = _subset(ctx, "test").reset_index(drop=True)
    tf = build_transforms(cfg.img_size, train=False)
    check = test.sample(min(256, len(test)), random_state=cfg.seed)
    xb = torch.stack([tf(load_rgb(p)) for p in check["path"]])
    with torch.inference_mode():   # small chunks keep the CPU memory low
        ref = np.concatenate([model(xb[i:i + 32]).numpy() for i in range(0, len(xb), 32)])
    out32, out16 = onnx_logits(fp32, xb.numpy()), onnx_logits(final, xb.numpy())
    parity = {
        "max_abs_diff_fp32": float(np.abs(ref - out32).max()),
        "max_abs_diff_fp16_weights": float(np.abs(ref - out16).max()),
        "prediction_agreement_fp16_weights": float((ref.argmax(1) == out16.argmax(1)).mean()),
        "n_images_checked": int(len(check)),
    }
    latency = benchmark_latency(final, cfg.img_size)
    info = {
        "model_name": "EfficientNet-B0 (transfer learning, fine-tuned)",
        "class_names": CLASS_NAMES,
        "input_size": cfg.img_size,
        "resize": "shorter side to input_size (bicubic), then centre crop",
        "mean": list(IMAGENET_MEAN),
        "std": list(IMAGENET_STD),
        "temperature": float(ctx.temperature),
        "confidence_threshold": APP_THRESHOLD,
        "metrics": {k: ctx.test_metrics.get(k) for k in
                    ["accuracy", "macro_f1", "balanced_accuracy", "top2_accuracy", "roc_auc_ovr_macro", "ece",
                     "accuracy_ci", "macro_f1_ci", "n"]},
        "per_class": pd.read_csv(ws.metrics_dir / "per_class_metrics.csv").round(4).to_dict("records"),
        "onnx_file_mb": round(final.stat().st_size / 1e6, 2),
        "cpu_latency_ms": latency,
        "parity": parity,
        "trained_on": time.strftime("%Y-%m-%d"),
        "dataset": "Garbage Images Dataset (2000/class), Kaggle: zlatan599/garbage-dataset-classification",
        "environment": environment_info(),
    }
    save_json(info, ws.model_dir / "model_info.json")
    shutil.copy(ws.fig("fig16_confusion_matrix.png"), ws.model_dir / "confusion_matrix.png")

    # Sample images for the demo: two confidently-correct test images per class + hardest cases
    preds = pd.read_csv(ws.metrics_dir / "test_predictions.csv")
    preds["path"] = test["path"].to_numpy()
    for f in ws.sample_dir.glob("*"):
        f.unlink()
    for cls in CLASS_NAMES:
        ok = preds[(preds["class_name"] == cls) & (preds["pred_class"] == cls)].sort_values("confidence", ascending=False)
        for k, (_, r) in enumerate(ok.iloc[[0, len(ok) // 2]].iterrows() if len(ok) > 1 else ok.iterrows(), 1):
            shutil.copy(r.path, ws.sample_dir / f"{cls}_{k:02d}{Path(r.path).suffix.lower()}")
    hard = preds[preds["pred_class"] != preds["class_name"]].sort_values("confidence").head(2)
    for k, (_, r) in enumerate(hard.iterrows(), 1):
        shutil.copy(r.path, ws.sample_dir / f"hard_case_{k:02d}_true-{r.class_name}{Path(r.path).suffix.lower()}")
    ctx.timings["export_s"] = time.time() - t0
    log(f"ONNX model: {info['onnx_file_mb']} MB, CPU latency {latency['median_ms']:.1f} ms/image, parity {parity}")
    return info


# =======================================================================================
# Package everything for the report
# =======================================================================================
def package_results(ctx: Context, ws: Workspace, zip_path=None, log=print) -> Path:
    save_json({"timings_s": ctx.timings, "environment": environment_info()}, ws.metrics_dir / "run_info.json")
    zip_path = Path(zip_path or ws.out_dir.parent / "results.zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for folder in [ws.fig_dir, ws.metrics_dir, ws.model_dir, ws.sample_dir]:
            for f in sorted(folder.rglob("*")):
                if f.is_file() and not f.name.endswith(".pt"):
                    zf.write(f, f.relative_to(ws.out_dir))
    log(f"Results packaged in {zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB)")
    return zip_path


def run_all(zip_path=None, data_dir=None, out_dir="outputs", device=None, workers=2, run_cnn=True,
            train_cfg=TrainConfig(), base_cfg=BaselineConfig(), split_cfg=SplitConfig(), log=print):
    """Run every step in order (used by scripts/run_pipeline.py)."""
    from .utils import get_device

    set_seed(train_cfg.seed)
    device = device or get_device()
    ws = Workspace(out_dir)
    root = acquire_data(zip_path, data_dir, extract_to=str(Path(out_dir) / "data"), log=log)
    ctx = data_understanding(root, ws, split_cfg, workers, log)
    visualise(ctx, ws, log)
    deep_embeddings(ctx, ws, device, workers=workers, log=log)
    tsne_map(ctx, ws, log=log)
    try:
        provenance(ctx, ws, workers=workers, log=log)
    except Exception as exc:  # needs internet access to Kaggle; optional
        log(f"Provenance check skipped: {exc}")
    split_data(ctx, ws, split_cfg, log)
    baselines(ctx, ws, device, base_cfg, run_cnn=run_cnn, workers=workers, log=log)
    train_final(ctx, ws, device, train_cfg, workers, log)
    evaluate_final(ctx, ws, device, train_cfg, workers, log=log)
    export_model(ctx, ws, train_cfg, log)
    return package_results(ctx, ws, log=log)


## Settings

In [ ]:
import sys, os, glob, json, shutil
from pathlib import Path
import numpy as np, pandas as pd
from IPython.display import Image, display, Markdown

sys.path.insert(0, "/content")
from waste_classifier import pipeline as pl, plots
from waste_classifier.config import CLASS_NAMES, TrainConfig, BaselineConfig, SplitConfig
from waste_classifier.utils import set_seed, get_device, environment_info

# ---- paths (change DRIVE_ZIP if the archive is somewhere else in your Google Drive) ----
DRIVE_ROOT = "/content/drive/MyDrive"
DRIVE_ZIP = f"{DRIVE_ROOT}/Auckland Institud/Artificial Intelligence/archive.zip"
DRIVE_OUT = f"{DRIVE_ROOT}/Auckland Institud/Artificial Intelligence/smart_waste_sorter_outputs"
OUT_DIR = "/content/outputs"

train_cfg, base_cfg, split_cfg = TrainConfig(), BaselineConfig(), SplitConfig()
WORKERS = os.cpu_count() or 2
set_seed(train_cfg.seed)
device = get_device()
if device.type != "cuda":      # no GPU: smaller batches to stay within the machine's RAM
    train_cfg.batch_size, base_cfg.cnn_batch_size = 32, 32
ws = pl.Workspace(OUT_DIR)

def show(name, width=950):
    """Display a saved figure inline."""
    display(Image(filename=str(ws.fig(name)), width=width))

def sync_to_drive():
    """Copy the outputs to Google Drive (so nothing is lost if the Colab session ends)."""
    if os.path.isdir(DRIVE_ROOT):
        shutil.copytree(OUT_DIR, DRIVE_OUT, dirs_exist_ok=True)
        print("Outputs copied to", DRIVE_OUT)

print("Device:", device)
print(json.dumps(environment_info(), indent=1))

## 1. Data acquisition and description (report section 1.1)
The Kaggle archive `archive.zip` is read from Google Drive and extracted to the fast local disk
of the Colab machine. If it is not found, it is downloaded directly from Kaggle with `kagglehub`.

In [ ]:
# Mount Google Drive (accept the permission pop-up). If it is declined or not answered
# within 2 minutes, the notebook continues and downloads the dataset from Kaggle instead.
try:
    from google.colab import drive
    drive.mount("/content/drive", timeout_ms=120_000)
except Exception as exc:
    print(f"Google Drive not mounted ({type(exc).__name__}) - using Kaggle instead.")
DRIVE_OK = os.path.isdir(DRIVE_ROOT)

zip_path = DRIVE_ZIP if DRIVE_OK and os.path.exists(DRIVE_ZIP) else None
if DRIVE_OK and zip_path is None:  # search the Drive for the archive (max. 3 folder levels deep)
    hits = [p for d in range(4) for p in glob.glob(DRIVE_ROOT + "/*" * d + "/archive.zip")]
    zip_path = hits[0] if hits else None
print("Archive:", zip_path or "not on Drive -> kagglehub download")
if zip_path:
    shutil.copy(zip_path, "/content/archive.zip")
    zip_path = "/content/archive.zip"
root = pl.acquire_data(zip_path=zip_path, extract_to="/content/data", use_kagglehub=True)
print("Dataset root:", root)

In [ ]:
ctx = pl.data_understanding(root, ws, split_cfg, workers=WORKERS)
s = ctx.summary
overview = pd.DataFrame({
    "Property": ["Total images", "Readable images", "Classes", "Resolution(s)", "File format(s)", "Colour mode(s)",
                 "Total size", "Median file size", "Exact duplicate files", "Images with a near-duplicate (pHash <= 4)",
                 "Near-duplicate pairs with different labels", "Largest / smallest class ratio"],
    "Value": [f"{s['n_images']:,}", f"{s['n_readable']:,}", s["n_classes"], s["resolutions"], s["formats"], s["modes"],
              f"{s['total_size_mb']:.1f} MB", f"{s['file_size_kb']['50%']:.1f} KB", s["exact_duplicate_files"],
              s["images_with_near_duplicate"]["<=4"], s["cross_class_pairs"]["<=4"], f"{s['imbalance_ratio_max_min']:.2f}"],
})
display(overview)
display(pd.Series(s["class_counts"], name="images").to_frame().assign(share=lambda d: (d.images / d.images.sum()).round(3)))
print("metadata.csv check:", json.dumps(s.get("metadata_check", {}), indent=1)[:1500])

In [ ]:
# Descriptive statistics of the numeric image variables (per class)
display(pd.read_csv(ws.metrics_dir / "image_statistics_overall.csv", index_col=0))
display(ctx.stats.groupby("class_name")[["brightness", "contrast", "saturation", "colourfulness", "sharpness",
                                          "entropy", "edge_density", "white_background", "file_size_kb"]].median().round(3))

## 2. Visualisation (report section 1.2)
Every image variable is visualised (class label, resolution, format, colour mode, file size,
brightness, contrast, saturation, colourfulness, sharpness, entropy, edge density, background,
colour channels and hue), followed by plots of the relationships between variables.

In [ ]:
figs = pl.visualise(ctx, ws)
for f in figs:
    display(Markdown(f"**{f.name}**"))
    display(Image(filename=str(f), width=950))

### Deep-feature map (t-SNE)
Each image is converted into a 1,280-number description by an ImageNet-pre-trained EfficientNet-B0,
and t-SNE projects these descriptions to 2-D. Clusters show which classes are visually distinct
and which overlap - a preview of the difficulty of the classification problem.

In [ ]:
pl.deep_embeddings(ctx, ws, device, workers=WORKERS)
pl.tsne_map(ctx, ws)
show("fig12_tsne_embeddings.png")

### Where did the images come from? (provenance check)
The Kaggle page does not document the original sources, so candidate public data sets
(TrashNet and two larger garbage collections on Kaggle) are downloaded and every image is
compared with ours through perceptual hashes. Matching images reveal the origin of the data.

In [ ]:
provenance_summary = pl.provenance(ctx, ws, workers=WORKERS)
display(provenance_summary)
show("fig25_provenance.png")

## 3. Data transformation (report section 2.2)
* Unreadable files, exact duplicates and files with conflicting labels are removed.
* A **stratified group split** (about 70 / 15 / 15 %) keeps the class proportions in each subset and
  puts near-duplicate images in the same subset, so the test set contains no copies of training images.
* Images are resized / centre-cropped to 224 x 224, scaled to [0, 1] and normalised with the ImageNet
  mean and standard deviation; training images are also randomly augmented.

In [ ]:
data = pl.split_data(ctx, ws, split_cfg)
show("fig13_split_distribution.png", 800)
show("fig14_augmentation_examples.png", 800)

## 4. Baseline models (report sections 2.1 and 2.4)
To justify the choice of model, simpler alternatives are trained on exactly the same split:
classical machine learning on hand-crafted features (k-NN, logistic regression, random forest, SVM),
a linear classifier on frozen ImageNet features, and a small CNN trained from scratch.

In [ ]:
baseline_results = pl.baselines(ctx, ws, device, base_cfg, run_cnn=True, workers=WORKERS)
display(baseline_results.round(4))

## 5. Training the final model: EfficientNet-B0 with transfer learning (report section 2.3)
Phase 1 trains only the new 6-class output layer on top of the frozen ImageNet backbone; phase 2
fine-tunes all layers with a smaller learning rate (AdamW, cosine schedule, label smoothing,
mixed precision on the GPU). Early stopping keeps the epoch with the best validation macro-F1.

In [ ]:
print(train_cfg)
model, history = pl.train_final(ctx, ws, device, train_cfg, workers=WORKERS)
display(history.round(4))
show("fig15_learning_curves.png")
sync_to_drive()

## 6. Evaluation on the held-out test set (report section 2.4)
Accuracy, precision, recall, F1 (per class and macro-averaged), confusion matrix, ROC and
precision-recall curves, 95% bootstrap confidence intervals, probability calibration
(temperature scaling), selective prediction, error analysis, Grad-CAM and robustness tests.

In [ ]:
metrics = pl.evaluate_final(ctx, ws, device, train_cfg, workers=WORKERS)
keys = ["accuracy", "accuracy_ci", "balanced_accuracy", "macro_precision", "macro_recall", "macro_f1", "macro_f1_ci",
        "top2_accuracy", "roc_auc_ovr_macro", "cohen_kappa", "mcc", "log_loss", "ece_before_temperature", "ece",
        "temperature", "accuracy_above_threshold", "coverage_above_threshold"]
display(pd.Series({k: metrics[k] for k in keys}, name="test set").to_frame())
display(pd.read_csv(ws.metrics_dir / "per_class_metrics.csv").round(4))
display(pd.read_csv(ws.metrics_dir / "most_confused_pairs.csv").round(4))
display(pd.read_csv(ws.metrics_dir / "model_comparison.csv").round(4))
for name in ["fig16_confusion_matrix.png", "fig17_per_class_metrics.png", "fig18_roc_pr_curves.png",
             "fig19_reliability_diagram.png", "fig20_confidence_analysis.png", "fig21_misclassified_examples.png",
             "fig22_gradcam.png", "fig23_robustness.png", "fig24_model_comparison.png"]:
    display(Markdown(f"**{name}**")); show(name)
sync_to_drive()

## 7. Export for the application (report section 3)
The model is exported to ONNX (weights stored in float16 to halve the file size) so the
application only needs `onnxruntime`, not PyTorch. The ONNX outputs are checked against PyTorch.

In [ ]:
info = pl.export_model(ctx, ws, train_cfg)
print(json.dumps({k: info[k] for k in ["onnx_file_mb", "cpu_latency_ms", "parity", "temperature", "metrics"]}, indent=1))
print(sorted(p.name for p in ws.sample_dir.iterdir()))

## 8. Save and download the results

In [ ]:
zip_file = pl.package_results(ctx, ws, zip_path="/content/results.zip")
sync_to_drive()
if os.path.isdir(DRIVE_ROOT):
    shutil.copy(zip_file, DRIVE_OUT + "/results.zip")
    print("results.zip saved in", DRIVE_OUT)
from google.colab import files
files.download(str(zip_file))

## (Only if needed) Recovery after a crash
If Colab stops with *"Your session crashed after using all available RAM"* **after** the training
cell has finished, the trained weights are already saved in `outputs/model/efficientnet_b0_best.pt`.
Do **not** run everything again: run the *Settings* cell above and then this cell. It reloads the
model, repeats the evaluation if it had not finished, exports the model and creates `results.zip`.

In [ ]:
import torch
from waste_classifier.models import create_efficientnet
data = pd.read_csv(ws.metrics_dir / "splits.csv")
root = pl.find_dataset_root("/content/data")
data["path"] = [str(root / p) for p in data["rel_path"]]
ctx = pl.Context(root=root, data=data)
model = create_efficientnet(len(CLASS_NAMES), pretrained=False, drop_rate=train_cfg.drop_rate,
                            drop_path_rate=train_cfg.drop_path_rate)
model.load_state_dict(torch.load(ws.model_dir / "efficientnet_b0_best.pt", map_location="cpu"))
ctx.model = model.eval()
if (ws.metrics_dir / "baseline_results.csv").exists():
    ctx.baselines = pd.read_csv(ws.metrics_dir / "baseline_results.csv")
if not (ws.metrics_dir / "test_metrics.json").exists():      # evaluation had not finished
    pl.evaluate_final(ctx, ws, device, train_cfg, workers=WORKERS)
ctx.test_metrics = json.loads((ws.metrics_dir / "test_metrics.json").read_text())
ctx.temperature = ctx.test_metrics["temperature"]
info = pl.export_model(ctx, ws, train_cfg)
zip_file = pl.package_results(ctx, ws, zip_path="/content/results.zip")
sync_to_drive()
from google.colab import files
files.download(str(zip_file))